# ICL-PILOT: generation, annotation and evaluation

**Notebook revision: Independent framework evaluation (2026-11-02); automatic source restoration and pinned UD downloads.**

Definitive ENNI study: 238 real children (190 TD, 48 SLI), 48 prompt pairs and 18 generation batches. Three pinned text models each generate 2,880 stories per stage at ten replicates: 17,280 narratives overall.

Use the complete prepared exemplars, fixed cross-story assignments and exact model-specific Stage 1 parents. Configuration and data sources are defined in `configs/generation/experiment.json`. Run-level seed 42, temperature 0.8, top-p 0.9, top-k 0 and 640 new tokens apply to both stages. Qwen thinking is disabled; the shared system sentence is folded into the user message before native chat formatting.

Jump to: [setup](#1-common-setup) | [generation](#2-generation) | [generation review](#review-generated-stories) | [CHAT conversion](#3-release-generation-model-segment-narratives-and-export-chat) | [shared preparation](#shared-annotation-preparation) | [TalkTag](#4-talktag-error-annotation) | [Batchalign UD](#batchalign-ud) | [domain features](#domain-features) | [processing exports](#5-qc-summary-and-exports) | [framework evaluation](#framework-evaluation). R analyses remain in their separate notebook.

Start with the smoke test to verify execution. Smoke outputs cannot enter scientific evaluation. See `docs/guides/colab_generation_package.md` for modes, dependencies and exports.

[Final text verification and retention](#annotation-retention)

Independent framework blocks added 2026-11-02; evaluation is disabled by default.

KIDEVAL English UD features added 2026-11-02; extraction is disabled by default.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = "https://github.com/shamira-venturini/ICL-PILOT.git"
try:
    import google.colab
    IS_COLAB = True
except ImportError:
    IS_COLAB = False
candidates = [Path.cwd(), *Path.cwd().parents, Path("/content/ICL-PILOT")]
REPO_DIR = next((p for p in candidates if (p / "pyproject.toml").is_file()), None)
if REPO_DIR is None:
    if not IS_COLAB:
        raise RuntimeError("Open this notebook from the accompanying ICL-PILOT checkout.")
    REPO_DIR = Path("/content/ICL-PILOT")
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
required = ["src/icl_pilot/pipeline.py", "src/icl_pilot/chat/annotation.py",
            "src/icl_pilot/generation/execution.py", "src/icl_pilot/generation/models.py",
            "src/icl_pilot/generation/inference.py", "configs/generation/models.json",
            "configs/annotation/talktag.json", "src/icl_pilot/chat/text_cleaning.py",
            "src/icl_pilot/chat/validation.py", "src/icl_pilot/chat/chatter_runtime.py",
            "configs/validation/chatter.json", "src/icl_pilot/chat/segmentation.py",
            "configs/segmentation/batchalign.json"]
missing = [name for name in required if not (REPO_DIR / name).is_file()]
if missing:
    raise RuntimeError("This checkout does not contain the notebook's updated helpers/configuration. "
                       "Use the accompanying project revision or upload that checkout. Missing: " + ", ".join(missing))
sys.path.insert(0, str(REPO_DIR / "src"))
print("Project:", REPO_DIR)

### Real ENNI annotation pilot

Choose `RUN_PROFILE="real_enni_pilot"` in Settings. The pilot contains the first
three four-year-old pairs from the definitive roster: TD 410, 450, 419 and
DLD/SLI 444, 480, 478. Six children provide 36 canonical narratives in six original
child CHAT files. This is a bounded processing pilot, not an accuracy estimate.

In a fresh Colab GPU runtime, upload the original `ICL-PILOT_smoke_bundle.zip`
and `ICL-PILOT_ENNI_annotation_pilot_inputs.zip`, then Run All. This route skips
generation and segmentation, retains original real utterance boundaries, and
runs shared cleaning, TalkTag, UD, preservation, Chatter, retention and export.
The original ENNI ZIP is included once as the recovery reference. Gillam remains
an external classifier dataset and is not needed for this annotation check.

`RUN_PROFILE="generation"` restores the existing generation/settings route.
Keep the completed synthetic smoke ZIP for the later annotation-quality comparison.
Download the resulting archive from Colab **Contents**; a copy is placed directly
in `/content` so it is visible without navigating inside the run directory.


## 1. Common setup

Use `SMOKE_TEST=True` and choose `SMOKE_TEST_JOBS` to limit the number of
target-story jobs. The current small-batch default is 12 jobs: **12 TD-like +
12 SLI-conditioned narratives**. Set the count to 1 for the original one-story
check. Test outputs are labelled and are not study data. Set `SMOKE_TEST=False`
for the complete experiment. This switch applies to new generation, not saved-input routes.

Choose the route and input paths below. Keep the default `REPLICATES=None`
to reproduce the configured ten replicates. A positive integer changes only
the new run; the frozen study configuration remains unchanged. With all 48
pairs, each replicate produces 288 TD-like and 288 SLI-conditioned narratives.

For `from_csv` / `convert_only` / `review_only`, supply cohort CSVs in `GENERATION_CSVS`.
Do not select both an aggregate CSV and its constituent cohort CSVs.
Use `review_only` for a separately runnable QC report on saved outputs.
It loads no models and leaves every story and bundle in the data.
For `from_chat`, `prepare_only` or `morphosyntax_only`, supply CHAT directories in `CHAT_INPUT_DIRS`; each may contain
`stage1_td` / `stage2_sli` subdirectories or a single collection of CHAT files.
Each input label must be unique, such as `age4` or `real_enni`.
Use `from_chat` for a combined real-plus-all-generators run; include real ENNI
once. Specify collection provenance in `CHAT_INPUT_METADATA`. Every annotation
run starts with the same shared cleaning/validation policy. `prepare_only`
performs that step without loading TalkTag or UD.
The full/from_csv/from_chat routes run TalkTag followed by Batchalign 3 UD
annotation by default. To tag saved real or synthetic CHAT without generation,
segmentation or TalkTag, select `MODE="morphosyntax_only"`, run common setup,
then jump to the Batchalign UD setup/run cells. `RUN_MORPHOSYNTAX=False` skips UD.

The prompt-ready exemplar CSV and its extraction audit travel with the checkout in
`configs/generation/inputs/`. Both Colab (including VS Code) and local Jupyter
use the checkout as `DATA_ROOT`, so the accompanying project revision needs no
separate input upload. Colab outputs persist under
`MyDrive/icl_pilot_generation_exports`. An explicit `ICL_PILOT_DATA_ROOT`
environment variable can select another tree containing the same configured
relative input path. For a separate CSV upload, set `GENERATION_INPUT_UPLOAD`
to its runtime path; the input cell copies it only to a missing destination,
verifies the hash and preserves conflicting existing inputs.

Use a fresh `RUN_ID` for every execution. For independent blocks, run this
common setup and the selected block; skipped generation cells do not load a model.

To continue interrupted generation, set `RESUME_RUN` to the original run directory
(local or mounted Drive) and retain its original model, smoke-test size and replicate settings.
Run setup/model blocks in order. The model loads before saved random states are restored.
Only one kernel may write a run. For subsequent CHAT/annotation work, use
`from_csv` or `from_chat` in a fresh processing run.


In [ ]:
from datetime import datetime, timezone
from icl_pilot.pipeline import stage_flags

RUN_PROFILE = 'generation'  # generation or real_enni_pilot
if RUN_PROFILE not in {"generation", "real_enni_pilot"}:
    raise ValueError("Select generation or real_enni_pilot.")
REAL_DATA_PILOT = RUN_PROFILE == "real_enni_pilot"

MODE = "full"  # full, generation_only, from_csv, convert_only, from_chat, prepare_only, morphosyntax_only, review_only
MODEL_KEY = "llama3_1_8b"  # llama3_1_8b, qwen3_8b, gemma2_9b; one model per fresh run
SMOKE_TEST = True  # Bounded test; set False for the full study run
SMOKE_TEST_JOBS = 12  # First 12 scheduled jobs: two replicates of six stories from the first pair
if REAL_DATA_PILOT:
    MODE = "from_chat"
    SMOKE_TEST = True
RUN_GENERATION, RUN_CHAT_CONVERSION, RUN_ANNOTATION = stage_flags(MODE)
RUN_MORPHOSYNTAX = RUN_ANNOTATION or MODE == "morphosyntax_only"  # Set False to stop after TalkTag
RESUME_RUN = None  # To continue: Path("/content/drive/MyDrive/icl_pilot_generation_exports/ORIGINAL_RUN_ID")
RESUME_GENERATION = RESUME_RUN is not None
REPLICATES = None  # None uses experiment.json; otherwise a positive integer
RUN_GENERATION_REVIEW = MODE in {"full", "generation_only", "from_csv", "convert_only", "review_only"}
REVIEW_JOURNALS = {}  # Optional {CSV_LABEL: Path("/saved/run/prompt_records.jsonl")}; otherwise look beside each CSV
DOWNLOAD_REVIEW_HTML = False  # Colab: download the offline review page after creating it
GENERATION_CSVS = {}  # e.g. {"age4": Path("/content/drive/MyDrive/run/4_year_old_rotating_story_generations.csv")}
CHAT_INPUT_DIRS = {}  # Multiple labelled collections; see shared preparation below
CHAT_INPUT_METADATA = {}  # Same labels: {"real_enni": {"source_type": "real", "corpus": "ENNI"}, ...}
ANNOTATION_STAGES = ["stage1_td", "stage2_sli"]
LEAN_ANNOTATION = True  # Verify final text, then remove successful-run intermediate CHAT copies
ORIGINAL_CORPUS_ARCHIVES = {}  # Optional {"ENNI": Path("/content/drive/MyDrive/ENNI.zip"), "Gillam": ...}
RETENTION_CONFIG_PATH = REPO_DIR / "configs/annotation/retention.json"

CONFIG_PATH = REPO_DIR / "configs/generation/experiment.json"
ROSTER_CSV = REPO_DIR / "configs/generation/master_frozen_roster_enni_2026-10-28.csv"
PREPARATION_CONFIG_PATH = REPO_DIR / "configs/annotation/preparation.json"
REVIEW_CONFIG_PATH = REPO_DIR / "configs/generation/review.json"
ANNOTATION_CONFIG_PATH = REPO_DIR / "configs/annotation/talktag.json"
MORPHOSYNTAX_CONFIG_PATH = REPO_DIR / "configs/annotation/batchalign.json"
MORPHOSYNTAX_ENV = Path("/content/icl_pilot_ud_env") if IS_COLAB else REPO_DIR / ".venv-morphosyntax"
MORPHOSYNTAX_MODELS_DIR = Path("/content/icl_pilot_ud_models") if IS_COLAB else REPO_DIR / ".cache/stanza_ud"
CHATTER_CONFIG_PATH = REPO_DIR / "configs/validation/chatter.json"
SEGMENTATION_CONFIG_PATH = REPO_DIR / "configs/segmentation/batchalign.json"
SEGMENTATION_ENV = Path("/content/icl_pilot_segmentation_env") if IS_COLAB else REPO_DIR / ".venv-segmentation"
CHATTER_EXECUTABLE = None  # Reuse/download pinned native CLI; optional explicit verified path
DATA_ROOT = Path(os.environ.get("ICL_PILOT_DATA_ROOT", str(REPO_DIR))).expanduser()
GENERATION_INPUT_UPLOAD = None  # Optional separate upload, e.g. Path("/content/story_unit_narratives_enni_2026-10-28.csv")
RUN_ID = ("enni_real_pilot_" if REAL_DATA_PILOT else ("enni_smoke_" if SMOKE_TEST and RUN_GENERATION else "enni_")) + (MODEL_KEY + "_" if RUN_GENERATION else "") + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
RUN_OUTPUT_DIR = Path(RESUME_RUN).expanduser() if RESUME_GENERATION else REPO_DIR / "data/generated/direct_generation" / RUN_ID
if RESUME_GENERATION:
    if not RUN_GENERATION:
        raise ValueError("RESUME_RUN is only for interrupted generation; use saved-input routes for later steps.")
    RUN_ID = RUN_OUTPUT_DIR.name
ANNOTATION_ENV = Path("/content/icl_pilot_talktag_env") if IS_COLAB else REPO_DIR / ".venv-talktag"

INSTALL_DEPENDENCIES = True
SAVE_EVERY = 25
SAVE_TO_DRIVE = IS_COLAB
DRIVE_EXPORT_ROOT = Path("/content/drive/MyDrive/icl_pilot_generation_exports")
MAKE_ARCHIVE = True
DOWNLOAD_ARCHIVE = False
RELEASE_COLAB_RUNTIME = False


# The real-data pilot uses the frozen input package; raw originals stay in its ZIP.
if REAL_DATA_PILOT:
    import hashlib, json, zipfile
    PILOT_INPUT_ZIP = (Path("/content/ICL-PILOT_ENNI_annotation_pilot_inputs.zip") if IS_COLAB
        else Path(os.environ.get("ICL_PILOT_ENNI_PILOT_ZIP",
            str(REPO_DIR / "data/inputs/ICL-PILOT_ENNI_annotation_pilot_inputs.zip"))))
    if not PILOT_INPUT_ZIP.is_file():
        raise FileNotFoundError("Upload ICL-PILOT_ENNI_annotation_pilot_inputs.zip to Colab, then rerun Settings.")
    if hashlib.sha256(PILOT_INPUT_ZIP.read_bytes()).hexdigest() != "e78db294df80707ff974d801d74c083d64561394f06136daaff32e516fc80a62":
        raise ValueError("Real ENNI pilot input package differs from the reviewed selection.")
    pilot_input_root = Path("/content/icl_pilot_enni_annotation_inputs") if IS_COLAB else REPO_DIR / ".cache/enni_annotation_pilot_inputs"
    with zipfile.ZipFile(PILOT_INPUT_ZIP) as _pilot_zip:
        _manifest_bytes = _pilot_zip.read("pilot_manifest.json")
        if hashlib.sha256(_manifest_bytes).hexdigest() != "aa37e2703feb424809d76d5cd6013aab2ec94823f9cc2bc4697701989f2eac07":
            raise ValueError("Real ENNI pilot selection manifest changed.")
        real_pilot_manifest = json.loads(_manifest_bytes)
        _expected_names = set(real_pilot_manifest["files"]) | {"pilot_manifest.json"}
        if len(_pilot_zip.namelist()) != len(_expected_names) or set(_pilot_zip.namelist()) != _expected_names:
            raise ValueError("Unexpected/duplicated real pilot package members.")
        _pilot_pending = []
        for _name in sorted(_expected_names):
            _relative = Path(_name)
            if _relative.is_absolute() or ".." in _relative.parts or "\\" in _name:
                raise ValueError("Unsafe real pilot package member.")
            _data = _pilot_zip.read(_name)
            _expected_sha = "aa37e2703feb424809d76d5cd6013aab2ec94823f9cc2bc4697701989f2eac07" if _name == "pilot_manifest.json" else real_pilot_manifest["files"][_name]
            if hashlib.sha256(_data).hexdigest() != _expected_sha:
                raise ValueError("Real pilot package checksum mismatch: " + _name)
            _target = pilot_input_root / _relative
            if _target.is_symlink() or any(parent.is_symlink() for parent in _target.parents):
                raise ValueError("Real pilot destination contains a symlink.")
            if _target.exists():
                if not _target.is_file() or hashlib.sha256(_target.read_bytes()).hexdigest() != _expected_sha:
                    raise ValueError("Existing real pilot input differs; preserved: " + str(_target))
            else:
                _pilot_pending.append((_target, _data))
        for _target, _data in _pilot_pending:
            _target.parent.mkdir(parents=True, exist_ok=True)
            _target.write_bytes(_data)
    _actual_chat = {path.relative_to(pilot_input_root).as_posix() for path in (pilot_input_root / "chat").rglob("*.cha")}
    if _actual_chat != {child["path"] for child in real_pilot_manifest["children"]}:
        raise ValueError("Unexpected CHAT files in the real pilot input directory; preserve and inspect them.")
    CHAT_INPUT_DIRS = {"real_enni": pilot_input_root / "chat"}
    CHAT_INPUT_METADATA = {"real_enni": {"source_type": "real", "corpus": "ENNI",
        "annotation_pilot": True, "selection_manifest_sha256": "aa37e2703feb424809d76d5cd6013aab2ec94823f9cc2bc4697701989f2eac07"}}
    ORIGINAL_CORPUS_ARCHIVES = {"ENNI": pilot_input_root / "ENNI.zip"}
    SAVE_TO_DRIVE = False
    print("Real ENNI pilot: 6 children (3 TD, 3 DLD), 36 narratives; original boundaries retained.")

print({"mode": MODE, "generation": RUN_GENERATION, "chat": RUN_CHAT_CONVERSION,
       "annotation": RUN_ANNOTATION, "ud_annotation": RUN_MORPHOSYNTAX, "smoke_test": SMOKE_TEST and (RUN_GENERATION or REAL_DATA_PILOT), "new_run": str(RUN_OUTPUT_DIR)})

In [ ]:
if RUN_CHAT_CONVERSION and sys.version_info < (3, 10):
    raise RuntimeError("Batchalign segmentation requires Python >=3.10. Use the Colab runtime.")
if (RUN_ANNOTATION or globals().get("RUN_MORPHOSYNTAX", False)) and sys.version_info < (3, 10):
    raise RuntimeError("TalkTag/Batchalign annotation requires Python >=3.10. Use the Colab runtime.")
if RUN_GENERATION and sys.version_info < (3, 9):
    raise RuntimeError("Generation with Transformers 4.57.1 requires Python >=3.9. Use the Colab runtime.")
if INSTALL_DEPENDENCIES:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--no-deps", "-e", str(REPO_DIR)], check=True)
    if RUN_GENERATION:
        subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "numpy", "pandas", "scipy", "tqdm",
                        "torch", "transformers==4.57.1", "accelerate", "bitsandbytes",
                        "huggingface_hub==0.36.0", "sentencepiece"], check=True)

In [ ]:
import json
from icl_pilot.pipeline import (begin_pipeline, checked_sources, record_stage,
                                convert_csv_inputs, annotation_inputs, annotation_collection_metadata, sync_run, package_run)

RESUME_GENERATION = globals().get("RESUME_GENERATION", False)
from icl_pilot.generation.recovery import sync_generation_checkpoint

drive_root = Path("/content/drive")
input_on_drive = (IS_COLAB and RUN_GENERATION and
                  (DATA_ROOT.resolve() == drive_root or drive_root in DATA_ROOT.resolve().parents))
resume_on_drive = IS_COLAB and RESUME_GENERATION and drive_root in RUN_OUTPUT_DIR.resolve().parents
saved_on_drive = IS_COLAB and any(drive_root in Path(value).expanduser().absolute().parents
    for value in [*GENERATION_CSVS.values(), *CHAT_INPUT_DIRS.values(), *globals().get("ORIGINAL_CORPUS_ARCHIVES", {}).values()])
if SAVE_TO_DRIVE or input_on_drive or resume_on_drive or saved_on_drive:
    from google.colab import drive
    if not Path("/content/drive/MyDrive").is_dir():
        drive.mount("/content/drive")
    if not Path("/content/drive/MyDrive").is_dir():
        raise RuntimeError("Google Drive is required for the selected inputs/exports but is not mounted.")

generation_csvs = checked_sources(GENERATION_CSVS) if MODE in {"from_csv", "convert_only", "review_only"} else {}
chat_roots = checked_sources(CHAT_INPUT_DIRS, directories=True) if MODE in {"from_chat", "prepare_only", "morphosyntax_only"} else {}
RUN_OUTPUT_DIR = begin_pipeline(RUN_OUTPUT_DIR, mode=MODE, csvs=generation_csvs, chat_dirs=chat_roots,
                                smoke_test=SMOKE_TEST and (RUN_GENERATION or globals().get("REAL_DATA_PILOT", False)), resume=RESUME_GENERATION)
for stage, enabled in (("generation", RUN_GENERATION),
                       ("generation_review", globals().get("RUN_GENERATION_REVIEW", RUN_GENERATION or MODE in {"from_csv", "convert_only", "review_only"})), ("chat_conversion", RUN_CHAT_CONVERSION),
                       ("annotation_preparation", RUN_ANNOTATION or MODE == "prepare_only"),
                       ("annotation", RUN_ANNOTATION),
                       ("morphosyntax", globals().get("RUN_MORPHOSYNTAX", False)), ("segmentation", RUN_CHAT_CONVERSION),
                       ("chat_validation_before", RUN_CHAT_CONVERSION or RUN_ANNOTATION),
                       ("chat_validation_after", RUN_ANNOTATION)):
    if not RESUME_GENERATION:
        record_stage(RUN_OUTPUT_DIR, stage, "pending" if enabled else "skipped")

def persist_run():
    if SAVE_TO_DRIVE and RUN_OUTPUT_DIR.resolve() != (DRIVE_EXPORT_ROOT / RUN_ID).resolve():
        target = sync_run(RUN_OUTPUT_DIR, DRIVE_EXPORT_ROOT / RUN_ID)
        print("Verified persistent copy:", target)

def persist_generation_checkpoint(source):
    if SAVE_TO_DRIVE:
        sync_generation_checkpoint(source, DRIVE_EXPORT_ROOT / RUN_ID)

print("Resuming pipeline:" if RESUME_GENERATION else "New pipeline run:", RUN_OUTPUT_DIR)
if globals().get("REAL_DATA_PILOT", False):
    from icl_pilot.generation.recovery import atomic_json
    _selection_path = RUN_OUTPUT_DIR / "real_annotation_pilot.json"
    atomic_json(_selection_path, real_pilot_manifest)
    record_stage(RUN_OUTPUT_DIR, "real_annotation_pilot", "complete",
        manifest=str(_selection_path), sha256=hashlib.sha256(_selection_path.read_bytes()).hexdigest(),
        input_children=real_pilot_manifest["child_count"], narratives=real_pilot_manifest["narrative_count"],
        clinical_counts=real_pilot_manifest["clinical_counts"], original_boundaries_retained=True)


In [ ]:
if RUN_GENERATION:
    import shutil
    from uuid import uuid4
    from icl_pilot.chat.annotation import sha256_file

    relative_input = Path(json.loads(CONFIG_PATH.read_text())["inputs"]["narratives_csv"])
    if relative_input.is_absolute() or ".." in relative_input.parts:
        raise ValueError("Generation input must stay inside DATA_ROOT.")
    input_csv = (DATA_ROOT / relative_input).resolve()
    if DATA_ROOT.resolve() not in input_csv.parents:
        raise ValueError("Generation input must stay inside DATA_ROOT.")
    upload = Path(GENERATION_INPUT_UPLOAD).expanduser() if GENERATION_INPUT_UPLOAD is not None else None
    input_action = "reused"
    if input_csv.exists():
        if not input_csv.is_file():
            raise FileExistsError(f"Expected an input file: {input_csv}")
        if upload is not None and upload.is_file() and sha256_file(upload) != sha256_file(input_csv):
            raise FileExistsError("Uploaded CSV differs from the existing input. Choose a fresh DATA_ROOT; "
                                  "the existing file has been preserved.")
    else:
        if upload is None or not upload.is_file():
            raise FileNotFoundError(f"Input missing: {input_csv}. Use the accompanying checkout with "
                                    "configs/generation/inputs/, or use VS Code's Upload to Colab "
                                    f"for {relative_input.name} and set GENERATION_INPUT_UPLOAD to its runtime path. "
                                    "Alternatively set DATA_ROOT to an existing input tree with the configured relative path.")
        input_csv.parent.mkdir(parents=True, exist_ok=True)
        temporary = input_csv.with_name(f".{input_csv.name}.{uuid4().hex}.copying")
        try:
            with upload.open("rb") as source, temporary.open("xb") as destination:
                shutil.copyfileobj(source, destination)
            if sha256_file(temporary) != sha256_file(upload):
                raise OSError("Generation input copy failed hash verification.")
            if input_csv.exists():
                raise FileExistsError(f"Input appeared during copying; preserved: {input_csv}")
            temporary.rename(input_csv)
        finally:
            if temporary.exists():
                temporary.unlink()
        input_action = "copied_and_verified"
    input_record = {"path": str(input_csv), "sha256": sha256_file(input_csv), "action": input_action}
    pipeline_path = RUN_OUTPUT_DIR / "pipeline_manifest.json"
    pipeline_record = json.loads(pipeline_path.read_text())
    pipeline_record.setdefault("generation_input", input_record)
    pipeline_record.setdefault("generation_input_checks", []).append(input_record)
    from icl_pilot.generation.recovery import atomic_json
    atomic_json(pipeline_path, pipeline_record)
    print("Generation input:", input_record)
    persist_run()
else:
    print("Exemplar input skipped; saved CSV/CHAT inputs will be used.")

## 2. Generation

This block validates the exemplar corpus, frozen ages and within-round
prompt/evaluation separation before loading the model. Inspect the job counts
before running the model-loading cell. The exemplar CSV path is relative to
`DATA_ROOT` (the checkout by default) and points into `configs/generation/inputs/`;
scaffolds and frozen prompts also come from `configs/generation/`.
The working text-only comparison shortlist is Llama 3.1 8B Instruct, Qwen3-8B
(non-thinking) and Gemma 2 9B IT. Select one with MODEL_KEY: llama3_1_8b,
qwen3_8b or gemma2_9b. All three use the shared loader and native chat adapters.
Run one checkpoint per fresh run; repeat the same smoke selection for each model.
Local software/template tests passed; full model inference still needs a Colab GPU test.
The vision condition is excluded from this study.
CHAT-to-CSV extraction removes annotator comments, pauses, xxx/yyy/www and &= codes
without guessing missing words or changing grammar. This notebook reads the resulting
prompt-ready CSV and validates/selects literal examples; it does not clean their text.
Leftover markers/comments fail preflight. The selected-text snapshot remains in
generation_design/prepared_exemplars.csv; removal counts belong to the extraction audit.
Both stages use those saved CSV texts; the Stage 1 parent remains literal.
Official model/tokenizer revisions are pinned in configs/generation/models.json.
Llama and Gemma require approved Hugging Face access. Supply an `HF_TOKEN`
through the runtime environment or an existing Hugging Face login; the loading cell
uses that authentication. The archived community conversion retains its original attribution.

Both reviewed prompts share the system message, ordered events, required content,
optional content, additional content constraints and full instruction list.
The exact system sentence is prepended to the user content for every model before native formatting.
The example guides wording, sentence construction, style and level of detail in both stages.
Stage 2 receives the exact Stage 1 narrative to rewrite with its selected exemplar.
Additional constraints are content-only. Age/status labels, ENNI/task labels, story/exemplar
IDs, titles and replicate IDs remain outside the templates and in validation/logs.
Story content and literal exemplar/parent text are preserved. No stage is asked to shorten its output.
The run records exact template hashes and messages. After pulling updates in an
existing session, restart the kernel and run from the top to load the new settings
and templates into a fresh run. Inspect the complete Stage 1 preview before inference.

Both stages read their output ceiling from `experiment.json`: currently **640 new
tokens per narrative**, shared across ages four through seven. Sampling also
uses shared configured values: **temperature 0.8, top-p 0.9 and top-k disabled (`top_k=0`)**
for both stages and every model condition.
The longest real story in those ages uses 568 tokens with Llama/Qwen and
598 with Gemma, including current exemplar preparation. The ceiling
excludes input tokens and permits earlier end-of-sequence stopping; it sets no
minimum or requested story length. The prompt journal records the ceiling and
actual generated-token count, terminal token and finish reason. Native stopping
IDs are preserved; formatted prompts are tokenized without adding special tokens again.
Qwen's template explicitly disables thinking. Other checkpoint/library decoding
defaults are recorded in model_runtime; common filtering/penalty policy remains pending.

Local CSV progress is saved every `SAVE_EVERY` complete jobs and at cohort
boundaries. After every completed generation call (Stage 1 or Stage 2), the
prompt/output journal and random-state checkpoint are saved; when
`SAVE_TO_DRIVE=True`, these recovery records are also synchronized to Drive.
The complete run folder is synchronized at cohort boundaries and other
persistence points. To continue an interrupted run, set `RESUME_RUN` to its
original directory and retain its model, smoke-test size and replicate settings.
Saved random states are restored after model loading, and verified completed
calls are reused. Only one kernel may write a run. See
[generation recovery](../docs/guides/generation_recovery.md). Saved CSVs can
enter conversion or annotation in a fresh processing run.

In [ ]:
if RUN_GENERATION:
    import pandas as pd
    import torch
    from icl_pilot.generation.config import prepare_colab_generation

    from icl_pilot.generation.randomness import initialise_run_randomness

    prepared = prepare_colab_generation(CONFIG_PATH, DATA_ROOT, RUN_OUTPUT_DIR,
                                        n_replicates=REPLICATES, smoke_test=SMOKE_TEST,
                                        smoke_test_jobs=SMOKE_TEST_JOBS if SMOKE_TEST else 1,
                                        model_key=MODEL_KEY, resume=RESUME_GENERATION)
    EXPERIMENT = prepared.config
    # Fresh run only, before model loading; resume restores states after loading.
    if not RESUME_GENERATION and globals().get("_SEEDED_RUN_ID") != RUN_ID:
        initialise_run_randomness(EXPERIMENT["master_seed"], torch=torch)
        _SEEDED_RUN_ID = RUN_ID
        from icl_pilot.generation.recovery import atomic_json, timestamp
        initial_manifest = json.loads(prepared.run_manifest_path.read_text())
        initial_manifest["randomness_initialization"] = {
            "master_seed": EXPERIMENT["master_seed"], "seed_policy": "run_level",
            "initialised_at_utc": timestamp(), "before_model_loading": True,
            "streams": ["python", "numpy", "torch_cpu", "torch_cuda_all"],
            "cuda_available": torch.cuda.is_available(),
        }
        atomic_json(prepared.run_manifest_path, initial_manifest)
    run_manifest_path = prepared.run_manifest_path
    run_df = prepared.run_df
    prompt_builder = prepared.prompt_builder
    MODEL_NAME, MODEL_REVISION = EXPERIMENT["model"]["id"], EXPERIMENT["model"]["revision"]
    DO_SAMPLE = EXPERIMENT["model"]["do_sample"]
    TEMPERATURE, TOP_P = EXPERIMENT["model"]["temperature"], EXPERIMENT["model"]["top_p"]
    TOP_K = EXPERIMENT["model"]["top_k"]
    print("Selected model:", EXPERIMENT["model"]["key"], MODEL_NAME, MODEL_REVISION)
    print(run_df.groupby("cohort_label", sort=False).agg(pairs=("pair_id", "nunique"),
        jobs=("target_story", "size"), rounds=("round_id", "nunique")).to_string())
    print(f"Total: {len(run_df)} TD-like + {len(run_df)} SLI-conditioned narratives")
    if SMOKE_TEST:
        print(f"Smoke test only: first {len(run_df)} scheduled jobs; do not use these outputs as study data.")
        print("The configured seed/order are retained; repeating the same setup may repeat the same outputs.")
    print("Configured replicates:", EXPERIMENT["n_replicates"], "| Prompt/evaluation separation: passed")
    print("Maximum new tokens: TD-like", EXPERIMENT["model"]["max_new_tokens_stage1"],
          "| SLI-conditioned", EXPERIMENT["model"]["max_new_tokens_stage2"])
    print("Sampling: temperature =", TEMPERATURE, "| top_p =", TOP_P, "| top_k =", TOP_K, "(0 disables top-k)")
    persist_run()
else:
    print("Generation setup skipped; saved inputs will be used.")

In [ ]:
if RUN_GENERATION:
    from icl_pilot.generation.inference import load_generation_runtime

    # One checkpoint on one GPU. Uses HF_TOKEN or the runtime's saved HF login.
    generation_runtime, runtime_metadata = load_generation_runtime(EXPERIMENT["model"])
    model, tokenizer = generation_runtime.model, generation_runtime.tokenizer
    print({key: runtime_metadata[key] for key in
           ("model_key", "model_id", "requested_revision", "gpu_name", "effective_compute_dtype",
            "native_stop_token_ids")})
    from icl_pilot.generation.execution import record_runtime_session
    runtime_session_id = record_runtime_session(run_manifest_path, runtime_metadata, resume=RESUME_GENERATION)
    # run_generation restores saved random streams after this model load.
    persist_run()


In [ ]:
def generate_from_messages(messages, *, max_new_tokens, row, stage, journal,
                           parent_record_id=None, stage1_text=None):
    return generation_runtime.generate(
        messages, max_new_tokens=max_new_tokens, row=row, stage=stage, journal=journal,
        parent_record_id=parent_record_id, stage1_text=stage1_text,
    )


In [ ]:
if RUN_GENERATION:
    example_row = run_df.iloc[0]
    packed_messages, preview = generation_runtime.render(prompt_builder.stage1_messages(example_row))
    print("Prompt template hashes:", prompt_builder.template_hashes)
    print("Instruction packaging:", EXPERIMENT["model"]["instruction_packaging"])
    print("Native template options:", EXPERIMENT["model"]["chat_template_kwargs"])
    print(preview)


In [ ]:
if RUN_GENERATION:
    from tqdm.auto import tqdm
    from icl_pilot.generation.execution import run_generation
    record_stage(RUN_OUTPUT_DIR, "generation", "running")
    try:
        paths = run_generation(prepared, generate_from_messages, run_id=RUN_ID,
            save_every=SAVE_EVERY, on_cohort_complete=lambda path: persist_run(), progress=tqdm,
            resume=globals().get("RESUME_GENERATION", False),
            on_checkpoint=globals().get("persist_generation_checkpoint"))
        generation_csvs = {path.stem.replace("_rotating_story_generations", ""): path for path in paths}
        record_stage(RUN_OUTPUT_DIR, "generation", "complete", csvs=[str(p) for p in paths])
    except BaseException:
        record_stage(RUN_OUTPUT_DIR, "generation", "interrupted; resume from completed-call checkpoint")
        raise
    finally:
        persist_run()

### Check for unchanged Stage 2 outputs

This read-only check compares saved Stage 1 and Stage 2 text after collapsing
whitespace. An unchanged result is flagged for review; it is preserved and
is not automatically resampled, discarded or edited to introduce errors.
A changed result alone does not establish DLD-like linguistic fidelity.
Likewise, no TalkTag error tags means no errors were tagged, not that
the narrative is error-free or that the clinical target was achieved.

In [ ]:
stage2_copy_summary = []
if RUN_GENERATION or MODE in {"from_csv", "convert_only", "review_only"}:
    import pandas as pd
    for label, path in generation_csvs.items():
        saved = pd.read_csv(path, dtype=str, keep_default_na=False)
        first = saved["stage1_td_story"].map(lambda text: " ".join(text.split()))
        second = saved["stage2_sli_story"].map(lambda text: " ".join(text.split()))
        unchanged = first.eq(second)
        count = int(unchanged.sum())
        stage2_copy_summary.append({"source_csv": str(path), "jobs": len(saved),
                                    "unchanged_after_whitespace": count})
        print(f"{label}: Stage 2 unchanged after whitespace normalization in {count}/{len(saved)} jobs.")
        if count:
            print("QC flag: inspect these transformations. Outputs and sampling are unchanged.")
            print(saved.loc[unchanged, ["pair_id", "target_story"]].head(10).to_string(index=False))
else:
    print("Stage comparison skipped; this route starts from CHAT files.")

<a id="review-generated-stories"></a>
### 2.1. Review generated stories — no exclusions

Run this block after generation or with saved `GENERATION_CSVS`. To review saved
outputs without loading models, converting CHAT or running annotation, choose
`MODE="review_only"`, run common setup, then jump here. Multiple CSV labels can
represent different generators or cohorts; select aggregate CSVs or their
constituent cohort files, avoiding overlapping jobs.

Every story remains in the generation, annotation and evaluation inputs. Flags
identify empty output, generation token-limit stops, Stage 2 unchanged after
whitespace normalization, possible template/role leakage, long same-word runs
and repeated phrase loops. Thresholds in `configs/generation/review.json` are
screening heuristics, not clinical norms. Ordinary child repetitions are retained.
Expected story content, exemplars and Stage 1 text are not treated as prompt
instructions. A flag is not evidence that a clinical target was achieved or missed.

Stopping reasons are read from `prompt_records.jsonl` beside each CSV, or from
explicit `REVIEW_JOURNALS` paths keyed by CSV label. Text/stage/identity and Stage 2
parent hashes must agree. Without a journal, stopping information is shown as
unknown; truncation is never guessed from length or punctuation. A natural end
token exactly at the budget does not receive a token-limit-stop flag.

The report contains **all stories**, a flagged-story CSV, flag counts by
source/generator/stage/cohort, source hashes and an offline `review.html` page.
The page supports search, filters, full Stage 1/Stage 2 text side by side, status
and notes. Download review notes as a separate CSV; they are not read by any
analysis. In Colab set `DOWNLOAD_REVIEW_HTML=True` or download the HTML through
the Files pane/Drive; open it in your browser. Reports travel with normal exports.
Rerunning this cell creates a new timestamped report and preserves prior reports.


In [ ]:
generation_review_manifest = None
if globals().get("RUN_GENERATION_REVIEW", RUN_GENERATION or MODE in {"from_csv", "convert_only", "review_only"}):
    from datetime import datetime, timezone
    from icl_pilot.generation.review import write_generation_review, DEFAULT_CONFIG as DEFAULT_REVIEW_CONFIG
    from icl_pilot.chat.annotation import sha256_file
    record_stage(RUN_OUTPUT_DIR, "generation_review", "running")
    try:
        review_output = RUN_OUTPUT_DIR / "generation_review" / datetime.now(timezone.utc).strftime("review_%Y%m%dT%H%M%S%fZ")
        generation_review_manifest = write_generation_review(generation_csvs, review_output,
            journals=globals().get("REVIEW_JOURNALS", {}),
            config_path=globals().get("REVIEW_CONFIG_PATH", DEFAULT_REVIEW_CONFIG))
        review_record = json.loads(generation_review_manifest.read_text())
        record_stage(RUN_OUTPUT_DIR, "generation_review", "complete", manifest=str(generation_review_manifest),
            manifest_sha256=sha256_file(generation_review_manifest), review_only=True, stories_excluded=0)
        print(json.dumps(review_record["summary"], indent=2))
        print("Offline review page:", review_output / "review.html")
        print("All stories and sampling are unchanged. Review notes are not applied to analyses.")
        if globals().get("DOWNLOAD_REVIEW_HTML", False):
            if IS_COLAB:
                from google.colab import files
                files.download(str(review_output / "review.html"))
            else:
                print("Open the HTML file above in your browser.")
    except BaseException:
        record_stage(RUN_OUTPUT_DIR, "generation_review", "failed_or_interrupted; inputs retained")
        raise
    finally:
        persist_run()
else:
    print("Generation review skipped; use saved generation CSVs to inspect stopping reasons and paired raw text.")


## 3. Release generation model, segment narratives and export CHAT

Batchalign 3 (pinned package 0.10.7) segments each complete CSV narrative before
CHAT utterances are written. Its English BERT word-boundary predictions are
applied to the original words/case, preserving grammatical forms and repetitions.
The isolated runtime defaults to CPU; no audio or speech-recognition models are used.

Existing orthographic preparation is logged. Predicted terminators replace prose
sentence punctuation; inline Unicode quotes do not create extra boundaries.
Protected CHAT notation stays anchored; conflicting scopes/terminators stop
conversion for review. Original CSVs and the exact Stage 1 parent stay unchanged.
Boundary actions, word offsets, punctuation changes and runtime/model hashes
are saved with each export. Failed model/coverage checks stop execution.

The next block requires Chatter before TalkTag; preservation QC and Chatter
are required again afterwards. Segmentation is not a grammatical correction or
a clinically validated boundary judgment. Review a small varied sample before
using new exports for linguistic analyses. Existing real CHAT boundaries remain unchanged.

Releasing the generator avoids retaining it during subsequent processing.

Known CHAT apostrophe formatting is checked before segmentation and audited again before Chatter. Raw generation text and hyphens are preserved.


In [ ]:
import gc
for name in ("generation_runtime", "model", "tokenizer"):
    globals().pop(name, None)
gc.collect()
if "torch" in globals() and torch.cuda.is_available():
    torch.cuda.empty_cache()
print("Generation model released; saved artifacts remain available.")

In [ ]:
# Known CHAT formatting rules are checked before conversion/Chatter.
if RUN_CHAT_CONVERSION:
    import hashlib, importlib
    from icl_pilot.chat import generated_chat
    from icl_pilot.generation.recovery import atomic_json
    source_path = REPO_DIR / "src/icl_pilot/chat/generated_chat.py"
    if Path(generated_chat.__file__).resolve() != source_path.resolve():
        raise RuntimeError("The imported CHAT converter belongs to a different checkout; restart with this notebook's project.")
    before_sha = hashlib.sha256(source_path.read_bytes()).hexdigest()
    upgraded = False
    # Compatibility with the exact, already-uploaded smoke bundle. Apply only
    # this reviewed four-line source change; never download or guess new code.
    if before_sha == "716f86a06a006920209e3b01aa6cbb0a44585526836e6c9a696bd45f3a7538a8":
        source = source_path.read_text(encoding="utf-8")
        insertion = (
            "    # CHAT rejects typographic single quotes even inside contractions (E256).\n"
            "    # Normalize their glyphs in derived text only, before segmentation, and\n"
            "    # retain each replacement alongside the unchanged source-text checksum.\n"
            "    substitute(r\"[‘’]\", lambda m: \"'\", \"ascii_apostrophe\")\n"
        )
        index = source.index('    substitute(r"\\(')
        fixed = source[:index] + insertion + source[index:]
        if hashlib.sha256(fixed.encode("utf-8")).hexdigest() != "7e006e63aba9d26f6a870fd91825c7faae15b47cfa41f3f5c18b27c72724fdbf":
            raise RuntimeError("The reviewed CHAT source update did not match its expected checksum.")
        source_path.write_text(fixed, encoding="utf-8")
        upgraded = True
    generated_chat = importlib.reload(generated_chat)
    probe, changes = generated_chat._normalize_generated_notation("they’re playing and he ‘goed’")
    if probe != "they're playing and he 'goed'" or sum(c["kind"] == "ascii_apostrophe" for c in changes) != 3:
        raise RuntimeError("This CHAT converter lacks the required audited apostrophe formatting rule.")
    report = dict(schema_version=1, status="ready", rule="curly_single_to_ascii_apostrophe",
        source_before_sha256=before_sha,
        source_after_sha256=hashlib.sha256(source_path.read_bytes()).hexdigest(),
        reviewed_compatibility_update_applied=upgraded,
        raw_generation_editing=False, hyphen_conversion=False)
    policy_path = RUN_OUTPUT_DIR / "chat_formatting_policy.json"
    atomic_json(policy_path, report)
    record_stage(RUN_OUTPUT_DIR, "chat_formatting_policy", "complete", manifest=str(policy_path),
                 sha256=hashlib.sha256(policy_path.read_bytes()).hexdigest())
    print("CHAT formatting ready: curly apostrophes are converted and audited; raw outputs are retained.")
else:
    print("CHAT conversion formatting skipped for this route.")


In [ ]:
if RUN_CHAT_CONVERSION:
    from icl_pilot.chat.segmentation import (
        BatchalignSegmenter, install_segmentation_runtime, load_segmentation_config)
    segmentation_settings = load_segmentation_config(SEGMENTATION_CONFIG_PATH)
    if INSTALL_DEPENDENCIES:
        segmentation_python = install_segmentation_runtime(SEGMENTATION_ENV, SEGMENTATION_CONFIG_PATH)
    else:
        segmentation_python = SEGMENTATION_ENV / ("Scripts/python.exe" if sys.platform == "win32" else "bin/python")
    segmentation_runtime = BatchalignSegmenter(segmentation_python, config_path=SEGMENTATION_CONFIG_PATH)
    print("Segmentation:", {key: segmentation_settings[key] for key in
          ("version", "model_id", "model_revision", "device", "policy")})
    record_stage(RUN_OUTPUT_DIR, "segmentation", "ready", config=segmentation_settings)
    persist_run()
else:
    print("Segmentation skipped: this mode does not convert generated CSVs.")


In [ ]:
if RUN_CHAT_CONVERSION:
    if "segmentation_runtime" not in globals():
        raise RuntimeError("Run the Batchalign segmentation setup block first.")
    record_stage(RUN_OUTPUT_DIR, "segmentation", "running")
    record_stage(RUN_OUTPUT_DIR, "chat_conversion", "running")
    try:
        # One model load processes all selected cohorts and both stages.
        chat_roots = convert_csv_inputs(generation_csvs, RUN_OUTPUT_DIR, roster_csv=ROSTER_CSV,
                                        segmenter=segmentation_runtime)
        record_stage(RUN_OUTPUT_DIR, "segmentation", "complete",
                     manifest=str(RUN_OUTPUT_DIR / "segmentation/segmentation_manifest.json"))
        record_stage(RUN_OUTPUT_DIR, "chat_conversion", "complete", roots={k: str(v) for k, v in chat_roots.items()})
        print("CHAT roots:", chat_roots)
    except BaseException:
        record_stage(RUN_OUTPUT_DIR, "segmentation", "failed_or_interrupted; review retained diagnostics")
        record_stage(RUN_OUTPUT_DIR, "chat_conversion", "failed; existing outputs retained")
        raise
    finally:
        persist_run()


In [ ]:
# Verify the actual derived CHAT and conversion audit before invoking Chatter.
if RUN_CHAT_CONVERSION:
    import csv, hashlib, json
    from icl_pilot.generation.recovery import atomic_json
    record_stage(RUN_OUTPUT_DIR, "chat_formatting", "running")
    try:
        checked = []
        for label, root in chat_roots.items():
            with (root / "chat_manifest.csv").open(encoding="utf-8", newline="") as handle:
                records = list(csv.DictReader(handle))
            if not records:
                raise ValueError("Missing CHAT conversion audit: " + label)
            for item in records:
                path = Path(item["output_path"])
                if root.resolve() not in path.resolve().parents:
                    raise ValueError("CHAT audit path falls outside its collection.")
                if hashlib.sha256(path.read_bytes()).hexdigest() != item["chat_sha256"]:
                    raise ValueError("Converted CHAT changed before its formatting check: " + str(path))
                for line in path.read_text(encoding="utf-8").splitlines():
                    if line.startswith("*") and any(char in line for char in "‘’"):
                        raise ValueError("Unconverted curly apostrophe before Chatter: " + str(path))
                if hashlib.sha256(Path(item["source_csv"]).read_bytes()).hexdigest() != item["source_csv_sha256"]:
                    raise ValueError("Raw generation CSV changed during CHAT conversion.")
                replacements = [c for c in json.loads(item["formatting_changes"]) if c["kind"] == "ascii_apostrophe"]
                if any(c["before"] not in ("‘", "’") or c["after"] != "'" for c in replacements):
                    raise ValueError("Unexpected apostrophe formatting audit.")
                checked.append(dict(collection=label, chat_file=str(path), chat_sha256=item["chat_sha256"],
                    source_text_sha256=item["source_text_sha256"], apostrophes_normalized=len(replacements)))
        report_path = RUN_OUTPUT_DIR / "chat_formatting_manifest.json"
        atomic_json(report_path, dict(schema_version=1, status="passed", files=checked,
            files_checked=len(checked), apostrophes_normalized=sum(c["apostrophes_normalized"] for c in checked),
            raw_generation_csvs_unchanged=True, chatter_validation="next_required_step"))
        record_stage(RUN_OUTPUT_DIR, "chat_formatting", "complete", manifest=str(report_path),
                     sha256=hashlib.sha256(report_path.read_bytes()).hexdigest())
        print("Pre-Chatter formatting passed:", len(checked), "files;",
              sum(c["apostrophes_normalized"] for c in checked), "apostrophes normalized.")
    except BaseException:
        record_stage(RUN_OUTPUT_DIR, "chat_formatting", "failed; Chatter/annotation blocked")
        raise
else:
    print("Generated CHAT formatting check skipped for this route.")


<a id="shared-annotation-preparation"></a>
### 3.1. Shared preparation of real and synthetic CHAT

For the joint study run, choose `MODE="from_chat"`, run common setup, then start
here. Prepare real ENNI once alongside every generator's saved CHAT, using
unique labels. A generator root may contain cohort directories; labels and all
nested relative paths survive preparation, TalkTag and UD. Select each file once;
overlapping input directories are rejected. Use `prepare_only` to run just this
cleaning and validation block without loading annotation models.

```python
CHAT_INPUT_DIRS = {
    "real_enni": REPO_DIR / "data/inputs/enni_verified_2026-10-28/chat_4_to_7",
    "llama": Path("/content/drive/MyDrive/LLAMA_RUN/chat"),
    "qwen": Path("/content/drive/MyDrive/QWEN_RUN/chat"),
    "gemma": Path("/content/drive/MyDrive/GEMMA_RUN/chat"),
}
CHAT_INPUT_METADATA = {
    "real_enni": {"source_type": "real", "corpus": "ENNI"},
    "llama": {"source_type": "synthetic", "generator": "llama3_1_8b"},
    "qwen": {"source_type": "synthetic", "generator": "qwen3_8b"},
    "gemma": {"source_type": "synthetic", "generator": "gemma2_9b"},
}
```

This common deletion-only policy removes all old dependent tiers, inline error
judgments, replacement/correction hints, glosses, alternative transcriptions and
annotator comments. Produced forms are retained, never replaced with target
forms. Metadata, main-tier boundaries, unknown speech, fragments, repetitions,
retracing, pauses, uncertainty, emphasis, actions, overlaps and language codes
are preserved. Structural clause markers and `bch`/`trn` utterance bookkeeping
are retained. Scope delimiters are unwrapped only when no retained code needs
them; whitespace exposed by deletion is removed only where CHAT requires it.

Originals remain unchanged. Collections explicitly marked `source_type="real"`
are referenced by file identity, checksum and available CHAT PID without duplicate
original-file backups. Synthetic/unspecified collections retain original snapshots.
With `LEAN_ANNOTATION=True`, keep one original ZIP per real corpus, referenced
by checksum rather than copied into every run. For Colab, set
`ORIGINAL_CORPUS_ARCHIVES` to your saved ZIP paths; these archives are not in the
Git checkout. Preparation checks required ZIPs before annotation starts. Available
working sources are still checked against their recorded checksums. Temporary
copies support model processing and failure recovery; successful-run copies are
removed only by the final retention block below. Final annotated CHAT, text
fingerprints, source identities, deletion audits and Chatter reports travel with
the run. Failed or incomplete runs retain their working files.
Standard Chatter structural and alignment validation is required before TalkTag
can start. Optional strict cross-utterance quotation/completion checks are not
enabled for real or synthetic files, because those relationships are outside the
study outcomes. Original markers and boundaries are preserved. Failures in the
required checks retain all prepared files and audits. Real ENNI is never resegmented.


In [ ]:
if RUN_ANNOTATION or MODE == "prepare_only":
    from icl_pilot.chat import validation as chat_validation
    from icl_pilot.chat.annotation_preparation import (prepare_annotation_collections,
        preparation_reference, require_prepared_annotations, DEFAULT_CONFIG as DEFAULT_PREPARATION_CONFIG)
    record_stage(RUN_OUTPUT_DIR, "annotation_preparation", "running")
    try:
        chatter_runtime = chat_validation.ensure_chatter(
            executable=globals().get("CHATTER_EXECUTABLE"),
            config_path=globals().get("CHATTER_CONFIG_PATH"), install=INSTALL_DEPENDENCIES)
        raw_annotation_inputs = annotation_inputs(chat_roots, ANNOTATION_STAGES)
        metadata = globals().get("CHAT_INPUT_METADATA", {})
        if RUN_CHAT_CONVERSION:
            metadata = {label: {"source_type": "synthetic", "processing_run_id": RUN_ID,
                **(metadata.get(label, {})),
                **({"generator": globals().get("MODEL_KEY"), "generation_run_id": RUN_ID}
                   if RUN_GENERATION else {})} for label in chat_roots}
        collection_metadata = annotation_collection_metadata(chat_roots, raw_annotation_inputs, metadata)
        if globals().get("LEAN_ANNOTATION", False):
            from icl_pilot.chat.corpus_archives import register_archives, DEFAULT_CONFIG as DEFAULT_RETENTION_CONFIG
            required_corpora = {item.get("corpus") for item in collection_metadata.values()
                if item.get("source_type") == "real"}
            if None in required_corpora:
                raise ValueError("Real collections require an explicit corpus in CHAT_INPUT_METADATA.")
            register_archives(RUN_OUTPUT_DIR / "original_corpus_archives.json", required=required_corpora,
                project_root=REPO_DIR, paths=globals().get("ORIGINAL_CORPUS_ARCHIVES", {}),
                config_path=globals().get("RETENTION_CONFIG_PATH", DEFAULT_RETENTION_CONFIG))
        preparation_manifest = prepare_annotation_collections(raw_annotation_inputs,
            RUN_OUTPUT_DIR / "annotation_preparation",
            config_path=globals().get("PREPARATION_CONFIG_PATH", DEFAULT_PREPARATION_CONFIG),
            collection_metadata=collection_metadata, validator=chatter_runtime,
            test_only=json.loads((RUN_OUTPUT_DIR / "pipeline_manifest.json").read_text())["test_only"])
        annotation_preparation_reference = preparation_reference(preparation_manifest)
        preparation_record = require_prepared_annotations(annotation_preparation_reference)
        prepared_annotation_inputs = {label: Path(item["prepared"])
            for label, item in preparation_record["collections"].items()}
        record_stage(RUN_OUTPUT_DIR, "annotation_preparation", "complete", **annotation_preparation_reference)
        print("Prepared collections:", {key: str(path) for key, path in prepared_annotation_inputs.items()})
    except BaseException as error:
        record_stage(RUN_OUTPUT_DIR, "annotation_preparation", "failed_or_interrupted; review retained audits/validation")
        # Include the actual validator codes in the exception, so VS Code logs
        # expose the cause without requiring another ZIP download.
        if getattr(error, "manifest_path", None):
            details = []
            try:
                failure_path = Path(error.manifest_path)
                failure = json.loads(failure_path.read_text())
                for command in failure.get("commands", []):
                    log_path = failure_path.parent / command["stdout"]
                    for line in log_path.read_text().splitlines():
                        event = json.loads(line)
                        if event.get("type") == "file" and event.get("status") != "valid":
                            messages = sorted({e.get("code", "unknown") + ": " + e.get("message", "") for e in event.get("errors", [])})
                            details.append(Path(event["file"]).name + " — " + "; ".join(messages))
                    stderr = (failure_path.parent / command["stderr"]).read_text().strip()
                    if stderr:
                        details.append(stderr[:1000])
                if not details:
                    details.append(json.dumps(failure.get("issues", [])))
            except (OSError, ValueError, KeyError, TypeError) as diagnostic_error:
                details.append("Could not read detailed report: " + str(diagnostic_error))
            error.args = (str(error) + "\nChatter diagnostic details:\n" + "\n".join(details[:12]),)
        raise
    finally:
        persist_run()
else:
    print("Shared preparation skipped for this route.")


In [ ]:
# Required structural gate: validate selected CHAT before installing/loading TalkTag.
if RUN_CHAT_CONVERSION or RUN_ANNOTATION:
    from icl_pilot.chat import validation as chat_validation
    record_stage(RUN_OUTPUT_DIR, "chat_validation_before", "running")
    try:
        chatter_runtime = chat_validation.ensure_chatter(
            executable=globals().get("CHATTER_EXECUTABLE"),
            config_path=globals().get("CHATTER_CONFIG_PATH"),
            install=INSTALL_DEPENDENCIES)
        if RUN_ANNOTATION:
            from icl_pilot.chat.annotation_preparation import require_prepared_annotations
            if not globals().get("annotation_preparation_reference"):
                raise RuntimeError("Run the shared annotation preparation block first.")
            preparation_record = require_prepared_annotations(annotation_preparation_reference)
            chat_validation_before = preparation_record["validation"]
            before_report = Path(chat_validation_before["manifest"])
        else:
            before_report = chat_validation.validate_chat_directories(
                chat_roots, RUN_OUTPUT_DIR / "validation/before_annotation", validator=chatter_runtime)
            chat_validation_before = chat_validation.validation_reference(before_report)
        record_stage(RUN_OUTPUT_DIR, "chat_validation_before", "complete", **chat_validation_before)
        print("Chatter passed before annotation:", before_report)
    except BaseException:
        record_stage(RUN_OUTPUT_DIR, "chat_validation_before", "failed; annotation blocked")
        raise
    finally:
        persist_run()
else:
    print("CHAT validation skipped: generation-only mode has no CHAT files.")


## 4. TalkTag error annotation

This block works with the converted CHAT files above or `CHAT_INPUT_DIRS` in
`from_chat` mode. Both synthetic stages are selected by default; change
`ANNOTATION_STAGES` to select one when using a root with stage subdirectories.
Run the shared preparation block first, including in `from_chat` mode.
All selected collections enter one TalkTag process with separate labels.

The checked source commit/options are in `configs/annotation/talktag.json`.
They specify a **new portable run**, the configured upstream source.
TalkTag is installed in an isolated environment, leaving generation dependencies
unchanged. No GitHub token is needed for its public source repository. If model
access requires credentials, supply `HF_TOKEN` through the runtime environment.

Each annotation input is copied into the new run. Wrapped CHI main tiers are
joined only in that copy. Prepared text and headers are kept for comparison; old dependent tiers and
inline linguistic annotations were removed by the shared preparation policy.
The preparation manifest and original-source lineage are retained. Exact TalkTag output is retained under `annotation/raw_talktag/`.
A deterministic postprocessor restores source capitalization and spacing in
`annotation/annotated/`, retaining inline tags at the same token boundaries.
Changed words, punctuation, order or boundaries stop processing for that file.
Final QC remains case-sensitive and checks coverage and source/tier preservation.
Failures retain outputs/logs for review. A nonzero debug limit is labelled partial.
Chatter checks are mandatory on source files and the exact input snapshot before TalkTag,
then on final postprocessed annotated files. Raw TalkTag structural diagnostics are retained.
A failed or incomplete check stops the pipeline; no validation errors are suppressed.
The source comparison also detects valid-but-changed headers, words and punctuation.
Error-tag correctness still requires linguistic review. The following block
adds fresh English UD morphology and dependencies using Batchalign 3. CLAN
measurement and feature extraction remain later stages.
To postprocess an existing retained batch without rerunning either model, see
[the recovery instructions](../docs/guides/colab_generation_package.md#conversion-and-annotation).

### Source-preservation policy

Automatically restore original words, deletions, order, punctuation and boundaries after TalkTag. Copy only uniquely aligned annotations onto the original source. Unmapped tags and unassessed words are recorded; they do not stop processing or exclude stories.

For a completed TalkTag run stopped by formatting QC: run the policy cell below, then the annotation-run cell and remaining cells. Keep the same kernel and run settings; generation and TalkTag inference are reused.


In [ ]:
# Automatic source restoration: undo annotation-side word edits and deletions.
if RUN_ANNOTATION:
    import hashlib, importlib
    from icl_pilot.chat import annotation as annotation_module
    from icl_pilot.generation.recovery import atomic_json
    source_path = REPO_DIR / "src/icl_pilot/chat/annotation.py"
    if Path(annotation_module.__file__).resolve() != source_path.resolve():
        raise RuntimeError("Annotation helpers belong to another checkout; use this notebook's project.")
    source_before = hashlib.sha256(source_path.read_bytes()).hexdigest()
    upgraded = False
    reviewed_updates = {
        "024647de64dc8d64183ea01d7ecc4471f2c03ddc1fa166eafaa96277e15aac13": [
                [
                        "_INLINE_ANNOTATION = re.compile(r\"\\[(?:\\*|::?|=)\\s+[^\\]]*\\]\")\n_ANNOTATED_TOKEN = re.compile(_INLINE_ANNOTATION.pattern + r\"|\\w+|[^\\w\\s]\")\n\n\ndef _restore_utterance_surface(original: str, annotated: str) -> str:\n    \"\"\"Keep annotation anchors while restoring source case and token spacing.\"\"\"\n    original_tokens, original_annotations = [], {}\n    for match in _ANNOTATED_TOKEN.finditer(original):\n        if _INLINE_ANNOTATION.fullmatch(match.group()):\n            original_annotations.setdefault(len(original_tokens), []).append(match)\n        else:\n            original_tokens.append(match)\n    annotated_tokens, annotations = [], {}\n    for match in _ANNOTATED_TOKEN.finditer(annotated):\n        if _INLINE_ANNOTATION.fullmatch(match.group()):\n            annotations.setdefault(len(annotated_tokens), []).append(match.group())\n        else:\n            annotated_tokens.append(match.group())\n    if ([match.group().lower() for match in original_tokens]\n            != [token.lower() for token in annotated_tokens]):\n        raise ValueError(\"Surface words, punctuation or their order changed beyond capitalization\")\n    if not original_tokens:\n        if original_annotations or annotations:\n            raise ValueError(\"Cannot anchor annotations to an empty utterance\")\n        return original\n    # Insert only new tags into the exact source string. Existing annotations\n    # must form an unchanged subsequence at each surface-token boundary.\n    insertions = {}\n    for anchor in sorted(set(annotations) | set(original_annotations)):\n        existing, index = original_annotations.get(anchor, []), 0\n        for tag in annotations.get(anchor, []):\n            if index < len(existing) and tag == existing[index].group():\n                index += 1\n                continue\n            if index < len(existing):\n                position, addition = existing[index].start(), tag + \" \"\n            elif existing:\n                position, addition = existing[-1].end(), \" \" + tag\n            elif anchor:\n                position, addition = original_tokens[anchor - 1].end(), \" \" + tag\n            else:\n                position, addition = original_tokens[0].start(), tag + \" \"\n            insertions.setdefault(position, []).append(addition)\n        if index != len(existing):\n            raise ValueError(\"An existing source annotation was removed, changed or moved\")\n    parts, cursor = [], 0\n    for position, additions in sorted(insertions.items()):\n        parts.extend([original[cursor:position], \"\".join(additions)])\n        cursor = position\n    parts.append(original[cursor:])\n    return \"\".join(parts)\n\n\ndef postprocess_annotated_chat(original: str, annotated: str, *, target: str = \"*CHI\") -> str:\n    \"\"\"Restore source orthography in a derived copy; never repair grammar.\"\"\"\n    source_lines = [line for line in _unfold_target(original, target).splitlines()\n                    if line.startswith(target + \":\")]\n    output_lines = _unfold_target(annotated, target).splitlines()\n    if len(source_lines) != sum(line.startswith(target + \":\") for line in output_lines):\n        raise ValueError(\"Target utterance count changed\")\n    sources, restored = iter(source_lines), []\n    for line in output_lines:\n        if line.startswith(target + \":\"):\n            source = next(sources)\n            body = _restore_utterance_surface(source.split(\":\", 1)[1], line.split(\":\", 1)[1])\n            restored.append(target + \":\" + body)\n        else:\n            restored.append(line)  # Headers and dependent tiers remain subject to strict QC.\n    return \"\\n\".join(restored) + \"\\n\"\n\n",
                        "_INLINE_ANNOTATION = re.compile(r\"\\[(?:\\*|::?|=)\\s+[^\\]]*\\]\")\n_RESTORATION_TOKEN = re.compile(\n    _INLINE_ANNOTATION.pattern + r\"|\\[[^\\[\\]]*\\]\"\n    r\"|\\(\\s*(?:\\.{1,3}|\\d+(?:\\.\\d+)?)\\s*\\)|&[=+~*][^\\s]+\"\n    r\"|(?:\\([A-Za-z]+\\))?\\w+(?:(?:['‘’:@-]|\\([A-Za-z]+\\))\\w*)*|[^\\w\\s]\"\n)\n\n\ndef _restoration_units(bodies):\n    tokens, annotations = [], []\n    for line_index, body in enumerate(bodies):\n        line_start = len(tokens)\n        for match in _RESTORATION_TOKEN.finditer(body):\n            item = {\"text\": match.group(), \"line\": line_index,\n                    \"start\": match.start(), \"end\": match.end()}\n            if _INLINE_ANNOTATION.fullmatch(match.group()):\n                annotations.append({**item, \"anchor\": len(tokens), \"leading\": len(tokens) == line_start})\n            elif match.group() not in {'\"', '“', '”'}:\n                tokens.append(item)\n    return tokens, annotations\n\n\ndef _is_restoration_word(token):\n    return (not token.startswith((\"[\", \"&\")) and not re.fullmatch(r\"\\([.\\d\\s]+\\)\", token)\n            and bool(re.search(r\"\\w\", token)))\n\n\ndef _restoration_key(token):\n    return token.lower().replace(\"‘\", \"'\").replace(\"’\", \"'\")\n\n\ndef _align_restoration_tokens(source, raw):\n    \"\"\"Return all optimal source anchors, refusing arbitrary repeated-word ties.\n\n    Text always comes from source. Only tags on unchanged, uniquely aligned\n    tokens are transferable; substituted/deleted words remain unassessed.\n    \"\"\"\n    a, b = [_restoration_key(x[\"text\"]) for x in source], [_restoration_key(x[\"text\"]) for x in raw]\n    n, m = len(a), len(b)\n    if a == b:\n        return [{j} for j in range(m)], [], 0\n\n    def substitution(i, j):\n        if a[i] == b[j]:\n            return 0\n        return 1 if _is_restoration_word(a[i]) and _is_restoration_word(b[j]) else 3\n\n    forward = [[0] * (m + 1) for _ in range(n + 1)]\n    for i in range(n + 1):\n        forward[i][0] = i\n    for j in range(m + 1):\n        forward[0][j] = j\n    for i in range(1, n + 1):\n        for j in range(1, m + 1):\n            forward[i][j] = min(forward[i - 1][j] + 1, forward[i][j - 1] + 1,\n                                forward[i - 1][j - 1] + substitution(i - 1, j - 1))\n    backward = [[0] * (m + 1) for _ in range(n + 1)]\n    for i in range(n, -1, -1):\n        backward[i][m] = n - i\n    for j in range(m, -1, -1):\n        backward[n][j] = m - j\n    for i in range(n - 1, -1, -1):\n        for j in range(m - 1, -1, -1):\n            backward[i][j] = min(backward[i + 1][j] + 1, backward[i][j + 1] + 1,\n                                 backward[i + 1][j + 1] + substitution(i, j))\n    total = forward[n][m]\n    possible = [set() for _ in range(m)]\n    for j in range(m):\n        for i in range(n + 1):\n            if forward[i][j] + 1 + backward[i][j + 1] == total:\n                possible[j].add(None)\n            if i < n and forward[i][j] + substitution(i, j) + backward[i + 1][j + 1] == total:\n                possible[j].add(i)\n    # One reproducible edit path is explanatory only. Tag transfer uses every\n    # optimal path above, so tie-breaking cannot move tags between repetitions.\n    edits, i, j = [], 0, 0\n    while i < n or j < m:\n        if i < n and j < m and backward[i][j] == substitution(i, j) + backward[i + 1][j + 1]:\n            if a[i] != b[j]:\n                edits.append({\"kind\": \"substitution_restored\", \"source_token\": i, \"raw_token\": j,\n                              \"source\": source[i][\"text\"], \"raw\": raw[j][\"text\"]})\n            i, j = i + 1, j + 1\n        elif i < n and backward[i][j] == 1 + backward[i + 1][j]:\n            edits.append({\"kind\": \"deletion_restored\", \"source_token\": i, \"source\": source[i][\"text\"]})\n            i += 1\n        else:\n            edits.append({\"kind\": \"model_insertion_removed\", \"raw_token\": j, \"raw\": raw[j][\"text\"]})\n            j += 1\n    return possible, edits, total\n\n\ndef _restore_source_bodies(originals, annotated):\n    source, existing_tags = _restoration_units(originals)\n    raw, raw_tags = _restoration_units(annotated)\n    possible, edits, distance = _align_restoration_tokens(source, raw)\n    confident = {j: next(iter(anchors)) for j, anchors in enumerate(possible)\n                 if len(anchors) == 1 and None not in anchors}\n    confident = {j: i for j, i in confident.items()\n                 if _restoration_key(source[i][\"text\"]) == _restoration_key(raw[j][\"text\"])}\n    source_annotations, new_annotations, discarded = {}, {}, []\n    for tag in existing_tags:\n        source_annotations.setdefault((tag[\"line\"], tag[\"anchor\"]), []).append(tag)\n    for tag in raw_tags:\n        previous = tag[\"anchor\"] - 1\n        if tag[\"leading\"] or previous not in confident:\n            discarded.append({\"tag\": tag[\"text\"], \"raw_utterance\": tag[\"line\"] + 1,\n                              \"raw_anchor\": tag[\"anchor\"], \"reason\": \"changed_inserted_or_ambiguous_anchor\"})\n            continue\n        index = confident[previous]\n        key = (source[index][\"line\"], index + 1)\n        new_annotations.setdefault(key, []).append(tag[\"text\"])\n    missing_existing = Counter()\n    for key, existing in source_annotations.items():\n        available = Counter(new_annotations.get(key, []))\n        for tag in existing:\n            if available[tag[\"text\"]]:\n                available[tag[\"text\"]] -= 1\n            else:\n                missing_existing[tag[\"text\"]] += 1\n    insertions, retained = {}, 0\n    for key, tags in sorted(new_annotations.items()):\n        line, anchor = key\n        existing = source_annotations.get(key, [])\n        index = 0\n        for tag in tags:\n            if index < len(existing) and tag == existing[index][\"text\"]:\n                index += 1\n        if index != len(existing):\n            discarded.extend({\"tag\": tag, \"source_utterance\": line + 1,\n                              \"reason\": \"existing_source_annotations_take_precedence\"}\n                             for tag in tags if tag not in {x[\"text\"] for x in existing})\n            continue\n        index = 0\n        for tag in tags:\n            if index < len(existing) and tag == existing[index][\"text\"]:\n                index += 1\n                continue\n            if missing_existing[tag]:\n                missing_existing[tag] -= 1\n                discarded.append({\"tag\": tag, \"source_utterance\": line + 1,\n                                  \"reason\": \"existing_source_annotation_moved\"})\n                continue\n            if index < len(existing):\n                position, addition = existing[index][\"start\"], tag + \" \"\n            elif existing:\n                position, addition = existing[-1][\"end\"], \" \" + tag\n            else:\n                position, addition = source[anchor - 1][\"end\"], \" \" + tag\n            insertions.setdefault(line, {}).setdefault(position, []).append(addition)\n            retained += 1\n    restored = []\n    for line, body in enumerate(originals):\n        parts, cursor = [], 0\n        for position, additions in sorted(insertions.get(line, {}).items()):\n            parts.extend([body[cursor:position], \"\".join(additions)])\n            cursor = position\n        parts.append(body[cursor:])\n        restored.append(\"\".join(parts))\n    assessed = set(confident.values())\n    unassessed = [{\"source_utterance\": token[\"line\"] + 1, \"token\": token[\"text\"],\n                   \"start\": token[\"start\"], \"end\": token[\"end\"]}\n                  for i, token in enumerate(source) if _is_restoration_word(token[\"text\"]) and i not in assessed]\n    word_edits = any(_is_restoration_word(edit.get(\"source\", \"\"))\n                     or _is_restoration_word(edit.get(\"raw\", \"\"))\n                     or edit.get(\"raw\", \"\").startswith((\"[*\", \"[:\", \"[=\")) for edit in edits)\n    return restored, {\"edit_distance\": distance, \"example_edit_path\": edits,\n                      \"model_word_edits\": word_edits,\n                      \"annotations_transferred\": retained, \"unmapped_annotations\": discarded,\n                      \"unassessed_source_words\": unassessed,\n                      \"annotation_coverage\": \"partial\" if word_edits or unassessed or discarded else \"complete\"}\n\n\ndef _restore_utterance_surface(original: str, annotated: str) -> str:\n    \"\"\"Always restore source text; copy only uniquely anchored annotation tags.\"\"\"\n    restored, _ = _restore_source_bodies([original], [annotated])\n    return restored[0]\n\n\ndef postprocess_annotated_chat(original: str, annotated: str, *, target: str = \"*CHI\", audit=None) -> str:\n    \"\"\"Rebuild from source, automatically undoing annotator edits and deletions.\"\"\"\n    source_lines = _unfold_target(original, target).splitlines()\n    indices = [i for i, line in enumerate(source_lines) if line.startswith(target + \":\")]\n    originals = [source_lines[i].split(\":\", 1)[1] for i in indices]\n    try:\n        raw_lines = _unfold_target(annotated, target).splitlines()\n    except ValueError:\n        raw_lines = annotated.splitlines()\n    raw = [line.split(\":\", 1)[1] for line in raw_lines if line.startswith(target + \":\")]\n    reports = []\n    if len(originals) == len(raw):\n        bodies = []\n        for utterance, (before, after) in enumerate(zip(originals, raw), 1):\n            restored, report = _restore_source_bodies([before], [after])\n            bodies.extend(restored)\n            for field in (\"unmapped_annotations\", \"unassessed_source_words\"):\n                for item in report[field]:\n                    if \"source_utterance\" in item:\n                        item[\"source_utterance\"] = utterance\n                    if \"raw_utterance\" in item:\n                        item[\"raw_utterance\"] = utterance\n            reports.append({\"source_utterance\": utterance, **report})\n    else:\n        bodies, report = _restore_source_bodies(originals, raw)\n        reports.append({\"source_utterance\": \"global_boundary_recovery\", **report})\n    for index, body in zip(indices, bodies):\n        source_lines[index] = target + \":\" + body\n    if audit is not None:\n        audit.update(source_utterances=len(originals), raw_utterances=len(raw),\n                     boundaries_restored=len(originals) != len(raw), utterances=reports,\n                     annotation_coverage=\"partial\" if any(r[\"annotation_coverage\"] == \"partial\" for r in reports) else \"complete\",\n                     raw_non_target_content_ignored=[line for line in raw_lines if not line.startswith(target + \":\")]\n                         != [line for line in source_lines if not line.startswith(target + \":\")])\n    return \"\\n\".join(source_lines) + \"\\n\"\n\n"
                ],
                [
                        "        before, raw_text = snapshot.read_text(), raw_path.read_text()\n        try:\n            after = postprocess_annotated_chat(before, raw_text, target=config[\"target_speaker\"])\n        except ValueError as error:\n",
                        "        before, raw_text = snapshot.read_text(), raw_path.read_text()\n        restoration = {}\n        try:\n            after = postprocess_annotated_chat(before, raw_text, target=config[\"target_speaker\"], audit=restoration)\n        except ValueError as error:\n"
                ],
                [
                        "            handle.write(after)\n        records.append({\"file\": relative, \"formatting_restored\": after != raw_text})\n        original_tiers = [_surface(x) for x in _target_utterances(before, config[\"target_speaker\"])]\n",
                        "            handle.write(after)\n        records.append({\"file\": relative, \"formatting_restored\": after != raw_text,\n                        \"source_restoration\": restoration})\n        original_tiers = [_surface(x) for x in _target_utterances(before, config[\"target_speaker\"])]\n"
                ],
                [
                        "        outputs.append({\"relative_path\": relative, \"sha256\": sha256_file(path),\n                        \"text_fingerprint\": fingerprint})\n    expected = {item[\"relative_path\"] for item in inventory}\n",
                        "        outputs.append({\"relative_path\": relative, \"sha256\": sha256_file(path),\n                        \"text_fingerprint\": fingerprint,\n                        \"annotation_coverage\": restoration[\"annotation_coverage\"]})\n    expected = {item[\"relative_path\"] for item in inventory}\n"
                ],
                [
                        "            \"tool_report_sha256\": sha256_file(report_path) if report_path.is_file() else None,\n            \"postprocessing\": {\"method\": \"restore_source_case_and_spacing\", \"version\": 1, \"files\": records},\n            \"qc_comparison\": \"case_sensitive_surface_tokens_and_utterance_boundaries\"}\n",
                        "            \"tool_report_sha256\": sha256_file(report_path) if report_path.is_file() else None,\n            \"postprocessing\": {\"method\": \"source_authoritative_automatic_restoration\", \"version\": 3,\n                               \"word_changes_and_deletions\": \"automatically_restored_from_source\",\n                               \"unmapped_tags\": \"recorded_without_arbitrary_assignment\",\n                               \"partial_annotation_files\": sum(item[\"annotation_coverage\"] == \"partial\" for item in outputs),\n                               \"files\": records},\n            \"qc_comparison\": \"case_sensitive_surface_tokens_and_utterance_boundaries\"}\n"
                ]
        ],
        "d4ac35ccab272fcda64d4ec2dd41e43438d5cdc1b33a3ed56f4b1355f439cf13": [
                [
                        "_INLINE_ANNOTATION = re.compile(r\"\\[(?:\\*|::?|=)\\s+[^\\]]*\\]\")\n_ANNOTATED_TOKEN = re.compile(_INLINE_ANNOTATION.pattern + r\"|\\w+|[^\\w\\s]\")\n\n\ndef _restore_utterance_surface(original: str, annotated: str) -> str:\n    \"\"\"Keep annotation anchors while restoring source case, spacing and quotes.\"\"\"\n    # TalkTag can omit prose double quotes. Align all other tokens exactly\n    # (apart from case), then insert new tags into the untouched source string.\n    # Final QC therefore still compares the original quotes and punctuation.\n    original_tokens, original_annotations = [], {}\n    for match in _ANNOTATED_TOKEN.finditer(original):\n        if _INLINE_ANNOTATION.fullmatch(match.group()):\n            original_annotations.setdefault(len(original_tokens), []).append(match)\n        elif match.group() not in {'\"', '“', '”'}:\n            original_tokens.append(match)\n    annotated_tokens, annotations = [], {}\n    for match in _ANNOTATED_TOKEN.finditer(annotated):\n        if _INLINE_ANNOTATION.fullmatch(match.group()):\n            annotations.setdefault(len(annotated_tokens), []).append(match.group())\n        elif match.group() not in {'\"', '“', '”'}:\n            annotated_tokens.append(match.group())\n    if ([match.group().lower() for match in original_tokens]\n            != [token.lower() for token in annotated_tokens]):\n        raise ValueError(\"Surface words, non-quotation punctuation or their order changed beyond capitalization and double-quote formatting\")\n    if not original_tokens:\n        if original_annotations or annotations:\n            raise ValueError(\"Cannot anchor annotations to an empty utterance\")\n        return original\n    # Insert only new tags into the exact source string. Existing annotations\n    # must form an unchanged subsequence at each surface-token boundary.\n    insertions = {}\n    for anchor in sorted(set(annotations) | set(original_annotations)):\n        existing, index = original_annotations.get(anchor, []), 0\n        for tag in annotations.get(anchor, []):\n            if index < len(existing) and tag == existing[index].group():\n                index += 1\n                continue\n            if index < len(existing):\n                position, addition = existing[index].start(), tag + \" \"\n            elif existing:\n                position, addition = existing[-1].end(), \" \" + tag\n            elif anchor:\n                position, addition = original_tokens[anchor - 1].end(), \" \" + tag\n            else:\n                position, addition = original_tokens[0].start(), tag + \" \"\n            insertions.setdefault(position, []).append(addition)\n        if index != len(existing):\n            raise ValueError(\"An existing source annotation was removed, changed or moved\")\n    parts, cursor = [], 0\n    for position, additions in sorted(insertions.items()):\n        parts.extend([original[cursor:position], \"\".join(additions)])\n        cursor = position\n    parts.append(original[cursor:])\n    return \"\".join(parts)\n\n\ndef postprocess_annotated_chat(original: str, annotated: str, *, target: str = \"*CHI\") -> str:\n    \"\"\"Restore source orthography and double quotes in a copy; never repair grammar.\"\"\"\n    source_lines = [line for line in _unfold_target(original, target).splitlines()\n                    if line.startswith(target + \":\")]\n    output_lines = _unfold_target(annotated, target).splitlines()\n    if len(source_lines) != sum(line.startswith(target + \":\") for line in output_lines):\n        raise ValueError(\"Target utterance count changed\")\n    sources, restored = iter(source_lines), []\n    for line in output_lines:\n        if line.startswith(target + \":\"):\n            source = next(sources)\n            body = _restore_utterance_surface(source.split(\":\", 1)[1], line.split(\":\", 1)[1])\n            restored.append(target + \":\" + body)\n        else:\n            restored.append(line)  # Headers and dependent tiers remain subject to strict QC.\n    return \"\\n\".join(restored) + \"\\n\"\n\n",
                        "_INLINE_ANNOTATION = re.compile(r\"\\[(?:\\*|::?|=)\\s+[^\\]]*\\]\")\n_RESTORATION_TOKEN = re.compile(\n    _INLINE_ANNOTATION.pattern + r\"|\\[[^\\[\\]]*\\]\"\n    r\"|\\(\\s*(?:\\.{1,3}|\\d+(?:\\.\\d+)?)\\s*\\)|&[=+~*][^\\s]+\"\n    r\"|(?:\\([A-Za-z]+\\))?\\w+(?:(?:['‘’:@-]|\\([A-Za-z]+\\))\\w*)*|[^\\w\\s]\"\n)\n\n\ndef _restoration_units(bodies):\n    tokens, annotations = [], []\n    for line_index, body in enumerate(bodies):\n        line_start = len(tokens)\n        for match in _RESTORATION_TOKEN.finditer(body):\n            item = {\"text\": match.group(), \"line\": line_index,\n                    \"start\": match.start(), \"end\": match.end()}\n            if _INLINE_ANNOTATION.fullmatch(match.group()):\n                annotations.append({**item, \"anchor\": len(tokens), \"leading\": len(tokens) == line_start})\n            elif match.group() not in {'\"', '“', '”'}:\n                tokens.append(item)\n    return tokens, annotations\n\n\ndef _is_restoration_word(token):\n    return (not token.startswith((\"[\", \"&\")) and not re.fullmatch(r\"\\([.\\d\\s]+\\)\", token)\n            and bool(re.search(r\"\\w\", token)))\n\n\ndef _restoration_key(token):\n    return token.lower().replace(\"‘\", \"'\").replace(\"’\", \"'\")\n\n\ndef _align_restoration_tokens(source, raw):\n    \"\"\"Return all optimal source anchors, refusing arbitrary repeated-word ties.\n\n    Text always comes from source. Only tags on unchanged, uniquely aligned\n    tokens are transferable; substituted/deleted words remain unassessed.\n    \"\"\"\n    a, b = [_restoration_key(x[\"text\"]) for x in source], [_restoration_key(x[\"text\"]) for x in raw]\n    n, m = len(a), len(b)\n    if a == b:\n        return [{j} for j in range(m)], [], 0\n\n    def substitution(i, j):\n        if a[i] == b[j]:\n            return 0\n        return 1 if _is_restoration_word(a[i]) and _is_restoration_word(b[j]) else 3\n\n    forward = [[0] * (m + 1) for _ in range(n + 1)]\n    for i in range(n + 1):\n        forward[i][0] = i\n    for j in range(m + 1):\n        forward[0][j] = j\n    for i in range(1, n + 1):\n        for j in range(1, m + 1):\n            forward[i][j] = min(forward[i - 1][j] + 1, forward[i][j - 1] + 1,\n                                forward[i - 1][j - 1] + substitution(i - 1, j - 1))\n    backward = [[0] * (m + 1) for _ in range(n + 1)]\n    for i in range(n, -1, -1):\n        backward[i][m] = n - i\n    for j in range(m, -1, -1):\n        backward[n][j] = m - j\n    for i in range(n - 1, -1, -1):\n        for j in range(m - 1, -1, -1):\n            backward[i][j] = min(backward[i + 1][j] + 1, backward[i][j + 1] + 1,\n                                 backward[i + 1][j + 1] + substitution(i, j))\n    total = forward[n][m]\n    possible = [set() for _ in range(m)]\n    for j in range(m):\n        for i in range(n + 1):\n            if forward[i][j] + 1 + backward[i][j + 1] == total:\n                possible[j].add(None)\n            if i < n and forward[i][j] + substitution(i, j) + backward[i + 1][j + 1] == total:\n                possible[j].add(i)\n    # One reproducible edit path is explanatory only. Tag transfer uses every\n    # optimal path above, so tie-breaking cannot move tags between repetitions.\n    edits, i, j = [], 0, 0\n    while i < n or j < m:\n        if i < n and j < m and backward[i][j] == substitution(i, j) + backward[i + 1][j + 1]:\n            if a[i] != b[j]:\n                edits.append({\"kind\": \"substitution_restored\", \"source_token\": i, \"raw_token\": j,\n                              \"source\": source[i][\"text\"], \"raw\": raw[j][\"text\"]})\n            i, j = i + 1, j + 1\n        elif i < n and backward[i][j] == 1 + backward[i + 1][j]:\n            edits.append({\"kind\": \"deletion_restored\", \"source_token\": i, \"source\": source[i][\"text\"]})\n            i += 1\n        else:\n            edits.append({\"kind\": \"model_insertion_removed\", \"raw_token\": j, \"raw\": raw[j][\"text\"]})\n            j += 1\n    return possible, edits, total\n\n\ndef _restore_source_bodies(originals, annotated):\n    source, existing_tags = _restoration_units(originals)\n    raw, raw_tags = _restoration_units(annotated)\n    possible, edits, distance = _align_restoration_tokens(source, raw)\n    confident = {j: next(iter(anchors)) for j, anchors in enumerate(possible)\n                 if len(anchors) == 1 and None not in anchors}\n    confident = {j: i for j, i in confident.items()\n                 if _restoration_key(source[i][\"text\"]) == _restoration_key(raw[j][\"text\"])}\n    source_annotations, new_annotations, discarded = {}, {}, []\n    for tag in existing_tags:\n        source_annotations.setdefault((tag[\"line\"], tag[\"anchor\"]), []).append(tag)\n    for tag in raw_tags:\n        previous = tag[\"anchor\"] - 1\n        if tag[\"leading\"] or previous not in confident:\n            discarded.append({\"tag\": tag[\"text\"], \"raw_utterance\": tag[\"line\"] + 1,\n                              \"raw_anchor\": tag[\"anchor\"], \"reason\": \"changed_inserted_or_ambiguous_anchor\"})\n            continue\n        index = confident[previous]\n        key = (source[index][\"line\"], index + 1)\n        new_annotations.setdefault(key, []).append(tag[\"text\"])\n    missing_existing = Counter()\n    for key, existing in source_annotations.items():\n        available = Counter(new_annotations.get(key, []))\n        for tag in existing:\n            if available[tag[\"text\"]]:\n                available[tag[\"text\"]] -= 1\n            else:\n                missing_existing[tag[\"text\"]] += 1\n    insertions, retained = {}, 0\n    for key, tags in sorted(new_annotations.items()):\n        line, anchor = key\n        existing = source_annotations.get(key, [])\n        index = 0\n        for tag in tags:\n            if index < len(existing) and tag == existing[index][\"text\"]:\n                index += 1\n        if index != len(existing):\n            discarded.extend({\"tag\": tag, \"source_utterance\": line + 1,\n                              \"reason\": \"existing_source_annotations_take_precedence\"}\n                             for tag in tags if tag not in {x[\"text\"] for x in existing})\n            continue\n        index = 0\n        for tag in tags:\n            if index < len(existing) and tag == existing[index][\"text\"]:\n                index += 1\n                continue\n            if missing_existing[tag]:\n                missing_existing[tag] -= 1\n                discarded.append({\"tag\": tag, \"source_utterance\": line + 1,\n                                  \"reason\": \"existing_source_annotation_moved\"})\n                continue\n            if index < len(existing):\n                position, addition = existing[index][\"start\"], tag + \" \"\n            elif existing:\n                position, addition = existing[-1][\"end\"], \" \" + tag\n            else:\n                position, addition = source[anchor - 1][\"end\"], \" \" + tag\n            insertions.setdefault(line, {}).setdefault(position, []).append(addition)\n            retained += 1\n    restored = []\n    for line, body in enumerate(originals):\n        parts, cursor = [], 0\n        for position, additions in sorted(insertions.get(line, {}).items()):\n            parts.extend([body[cursor:position], \"\".join(additions)])\n            cursor = position\n        parts.append(body[cursor:])\n        restored.append(\"\".join(parts))\n    assessed = set(confident.values())\n    unassessed = [{\"source_utterance\": token[\"line\"] + 1, \"token\": token[\"text\"],\n                   \"start\": token[\"start\"], \"end\": token[\"end\"]}\n                  for i, token in enumerate(source) if _is_restoration_word(token[\"text\"]) and i not in assessed]\n    word_edits = any(_is_restoration_word(edit.get(\"source\", \"\"))\n                     or _is_restoration_word(edit.get(\"raw\", \"\"))\n                     or edit.get(\"raw\", \"\").startswith((\"[*\", \"[:\", \"[=\")) for edit in edits)\n    return restored, {\"edit_distance\": distance, \"example_edit_path\": edits,\n                      \"model_word_edits\": word_edits,\n                      \"annotations_transferred\": retained, \"unmapped_annotations\": discarded,\n                      \"unassessed_source_words\": unassessed,\n                      \"annotation_coverage\": \"partial\" if word_edits or unassessed or discarded else \"complete\"}\n\n\ndef _restore_utterance_surface(original: str, annotated: str) -> str:\n    \"\"\"Always restore source text; copy only uniquely anchored annotation tags.\"\"\"\n    restored, _ = _restore_source_bodies([original], [annotated])\n    return restored[0]\n\n\ndef postprocess_annotated_chat(original: str, annotated: str, *, target: str = \"*CHI\", audit=None) -> str:\n    \"\"\"Rebuild from source, automatically undoing annotator edits and deletions.\"\"\"\n    source_lines = _unfold_target(original, target).splitlines()\n    indices = [i for i, line in enumerate(source_lines) if line.startswith(target + \":\")]\n    originals = [source_lines[i].split(\":\", 1)[1] for i in indices]\n    try:\n        raw_lines = _unfold_target(annotated, target).splitlines()\n    except ValueError:\n        raw_lines = annotated.splitlines()\n    raw = [line.split(\":\", 1)[1] for line in raw_lines if line.startswith(target + \":\")]\n    reports = []\n    if len(originals) == len(raw):\n        bodies = []\n        for utterance, (before, after) in enumerate(zip(originals, raw), 1):\n            restored, report = _restore_source_bodies([before], [after])\n            bodies.extend(restored)\n            for field in (\"unmapped_annotations\", \"unassessed_source_words\"):\n                for item in report[field]:\n                    if \"source_utterance\" in item:\n                        item[\"source_utterance\"] = utterance\n                    if \"raw_utterance\" in item:\n                        item[\"raw_utterance\"] = utterance\n            reports.append({\"source_utterance\": utterance, **report})\n    else:\n        bodies, report = _restore_source_bodies(originals, raw)\n        reports.append({\"source_utterance\": \"global_boundary_recovery\", **report})\n    for index, body in zip(indices, bodies):\n        source_lines[index] = target + \":\" + body\n    if audit is not None:\n        audit.update(source_utterances=len(originals), raw_utterances=len(raw),\n                     boundaries_restored=len(originals) != len(raw), utterances=reports,\n                     annotation_coverage=\"partial\" if any(r[\"annotation_coverage\"] == \"partial\" for r in reports) else \"complete\",\n                     raw_non_target_content_ignored=[line for line in raw_lines if not line.startswith(target + \":\")]\n                         != [line for line in source_lines if not line.startswith(target + \":\")])\n    return \"\\n\".join(source_lines) + \"\\n\"\n\n"
                ],
                [
                        "        before, raw_text = snapshot.read_text(), raw_path.read_text()\n        try:\n            after = postprocess_annotated_chat(before, raw_text, target=config[\"target_speaker\"])\n        except ValueError as error:\n",
                        "        before, raw_text = snapshot.read_text(), raw_path.read_text()\n        restoration = {}\n        try:\n            after = postprocess_annotated_chat(before, raw_text, target=config[\"target_speaker\"], audit=restoration)\n        except ValueError as error:\n"
                ],
                [
                        "            handle.write(after)\n        records.append({\"file\": relative, \"formatting_restored\": after != raw_text})\n        original_tiers = [_surface(x) for x in _target_utterances(before, config[\"target_speaker\"])]\n",
                        "            handle.write(after)\n        records.append({\"file\": relative, \"formatting_restored\": after != raw_text,\n                        \"source_restoration\": restoration})\n        original_tiers = [_surface(x) for x in _target_utterances(before, config[\"target_speaker\"])]\n"
                ],
                [
                        "        outputs.append({\"relative_path\": relative, \"sha256\": sha256_file(path),\n                        \"text_fingerprint\": fingerprint})\n    expected = {item[\"relative_path\"] for item in inventory}\n",
                        "        outputs.append({\"relative_path\": relative, \"sha256\": sha256_file(path),\n                        \"text_fingerprint\": fingerprint,\n                        \"annotation_coverage\": restoration[\"annotation_coverage\"]})\n    expected = {item[\"relative_path\"] for item in inventory}\n"
                ],
                [
                        "            \"tool_report_sha256\": sha256_file(report_path) if report_path.is_file() else None,\n            \"postprocessing\": {\"method\": \"restore_source_case_spacing_and_double_quotes\", \"version\": 2, \"files\": records},\n            \"qc_comparison\": \"case_sensitive_surface_tokens_and_utterance_boundaries\"}\n",
                        "            \"tool_report_sha256\": sha256_file(report_path) if report_path.is_file() else None,\n            \"postprocessing\": {\"method\": \"source_authoritative_automatic_restoration\", \"version\": 3,\n                               \"word_changes_and_deletions\": \"automatically_restored_from_source\",\n                               \"unmapped_tags\": \"recorded_without_arbitrary_assignment\",\n                               \"partial_annotation_files\": sum(item[\"annotation_coverage\"] == \"partial\" for item in outputs),\n                               \"files\": records},\n            \"qc_comparison\": \"case_sensitive_surface_tokens_and_utterance_boundaries\"}\n"
                ]
        ],
        "7d1c89eb5cc1f70a4b46c7b89597d37d741ab1f858674cbd99aa92cb39351620": [
                [
                        "                  for i, token in enumerate(source) if _is_restoration_word(token[\"text\"]) and i not in assessed]\n    return restored, {\"edit_distance\": distance, \"example_edit_path\": edits,\n                      \"annotations_transferred\": retained, \"unmapped_annotations\": discarded,\n                      \"unassessed_source_words\": unassessed,\n                      \"annotation_coverage\": \"partial\" if unassessed or discarded else \"complete\"}\n\n",
                        "                  for i, token in enumerate(source) if _is_restoration_word(token[\"text\"]) and i not in assessed]\n    word_edits = any(_is_restoration_word(edit.get(\"source\", \"\"))\n                     or _is_restoration_word(edit.get(\"raw\", \"\"))\n                     or edit.get(\"raw\", \"\").startswith((\"[*\", \"[:\", \"[=\")) for edit in edits)\n    return restored, {\"edit_distance\": distance, \"example_edit_path\": edits,\n                      \"model_word_edits\": word_edits,\n                      \"annotations_transferred\": retained, \"unmapped_annotations\": discarded,\n                      \"unassessed_source_words\": unassessed,\n                      \"annotation_coverage\": \"partial\" if word_edits or unassessed or discarded else \"complete\"}\n\n"
                ]
        ]
}
    if source_before in reviewed_updates:
        source = source_path.read_text(encoding="utf-8")
        reviewed_replacements = reviewed_updates[source_before]
        for old, new in reviewed_replacements:
            if source.count(old) != 1:
                raise RuntimeError("The reviewed annotation source update has unexpected context.")
            source = source.replace(old, new, 1)
        if hashlib.sha256(source.encode("utf-8")).hexdigest() != "045672497b867cf2d712a26ecc525b1d0fa719022d76c84be37a19b911e2213f":
            raise RuntimeError("The reviewed annotation source update failed its checksum.")
        source_path.write_text(source, encoding="utf-8")
        upgraded = True
    annotation_module = importlib.reload(annotation_module)
    source_probe = "*CHI:\t“no running” .\n"
    if annotation_module.postprocess_annotated_chat(source_probe, "*CHI:\tno running.\n") != source_probe:
        raise RuntimeError("The source double-quote restoration rule is unavailable.")
    coverage_probe = {}
    restored_probe = annotation_module.postprocess_annotated_chat(source_probe,
        "*CHI:\tyes [* s:r:gc:pro] running.\n", audit=coverage_probe)
    if restored_probe != source_probe or coverage_probe.get("annotation_coverage") != "partial":
        raise RuntimeError("Changed words must be restored and unmapped tags recorded.")
    if annotation_module.postprocess_annotated_chat(source_probe, "*CHI:\trunning.\n") != source_probe:
        raise RuntimeError("Deleted words must be restored from source automatically.")
    insertion_probe = {}
    if annotation_module.postprocess_annotated_chat(source_probe, "*CHI:\tno really running.\n", audit=insertion_probe) != source_probe or insertion_probe.get("annotation_coverage") != "partial":
        raise RuntimeError("Model-inserted words must be removed and annotation coverage recorded.")
    annotate_chat_directory = annotation_module.annotate_chat_directory
    postprocess_annotation_run = annotation_module.postprocess_annotation_run
    policy_path = RUN_OUTPUT_DIR / "annotation_preservation_policy.json"
    atomic_json(policy_path, dict(schema_version=1, status="ready", postprocessing_version=3,
        restored_features=["words", "word_order", "utterance_boundaries", "case", "punctuation", "source_layout"],
        words_and_boundaries="source_authoritative_automatic_restoration", source_before_sha256=source_before,
        source_after_sha256=hashlib.sha256(source_path.read_bytes()).hexdigest(),
        reviewed_compatibility_update_applied=upgraded, raw_talktag_outputs_edited=False))
    record_stage(RUN_OUTPUT_DIR, "annotation_preservation_policy", "complete", manifest=str(policy_path),
                 sha256=hashlib.sha256(policy_path.read_bytes()).hexdigest())
    print("Automatic source restoration ready: annotator word edits and deletions are restored; unmapped tags are audited.")
else:
    print("TalkTag preservation setup skipped for this route.")


In [ ]:
if RUN_ANNOTATION:
    from icl_pilot.chat.validation import require_validated_report
    if not globals().get("chat_validation_before"):
        raise RuntimeError("Run the required pre-annotation CHAT validation block first.")
    require_validated_report(chat_validation_before)
    from icl_pilot.chat.annotation import (load_annotation_config, install_annotation_runtime,
                                      annotation_runtime_metadata, annotate_chat_directory)
    annotation_settings = load_annotation_config(ANNOTATION_CONFIG_PATH)
    from icl_pilot.chat.annotation_preparation import require_prepared_annotations
    if not globals().get("annotation_preparation_reference"):
        raise RuntimeError("Run shared annotation preparation before TalkTag.")
    preparation_record = require_prepared_annotations(annotation_preparation_reference)
    selected_chat_inputs = {label: Path(item["prepared"]) for label, item in preparation_record["collections"].items()}
    print({label: str(path) for label, path in selected_chat_inputs.items()})
    if INSTALL_DEPENDENCIES:
        annotation_python = install_annotation_runtime(ANNOTATION_ENV, annotation_settings)
    else:
        annotation_python = ANNOTATION_ENV / ("Scripts/python.exe" if sys.platform == "win32" else "bin/python")
    annotation_runtime = annotation_runtime_metadata(annotation_python, annotation_settings)
    print("TalkTag runtime:", annotation_runtime)

In [ ]:
if RUN_ANNOTATION:
    from datetime import datetime, timezone
    from icl_pilot.chat.annotation import annotate_chat_directory, postprocess_annotation_run
    annotation_manifests = []
    record_stage(RUN_OUTPUT_DIR, "annotation", "running")
    try:
        original_manifest = RUN_OUTPUT_DIR / "annotation/annotation_manifest.json"
        if original_manifest.is_file():
            prior = json.loads(original_manifest.read_text())
            completed_status = prior.get("status") in {"annotation_complete", "partial_debug"} or prior.get("status", "").startswith("failed_or_interrupted")
            if prior.get("exit_code") != 0 or not completed_status:
                raise RuntimeError("This annotation run is not a completed TalkTag process awaiting QC recovery; retained files are unchanged.")
            if prior.get("config") != annotation_settings or prior.get("input_preparation") != globals().get("annotation_preparation_reference"):
                raise RuntimeError("Saved annotation settings or prepared inputs differ; use a separate processing run.")
            reviewed_name = datetime.now(timezone.utc).strftime("postprocessed_%Y%m%dT%H%M%S%fZ")
            # This verifies every retained raw checksum and re-runs preservation
            # and Chatter. It never invokes TalkTag or rewrites the old manifest.
            manifest_path = postprocess_annotation_run(original_manifest,
                output_name=reviewed_name, validator=globals().get("chatter_runtime"))
            print("Rechecked retained TalkTag outputs; no model inference was repeated.")
        else:
            # One invocation processes all selected cohorts/stages, loading TalkTag once.
            manifest_path = annotate_chat_directory(selected_chat_inputs, RUN_OUTPUT_DIR / "annotation",
                config=annotation_settings, python=annotation_python, runtime=annotation_runtime,
                validator=globals().get("chatter_runtime"),
                preparation_reference=annotation_preparation_reference)
        annotation_manifests.append(manifest_path)
        print("Annotation and QC:", manifest_path)
        annotation_records = [json.loads(p.read_text()) for p in annotation_manifests]
        statuses = [item["status"] for item in annotation_records]
        status = "partial_debug" if "partial_debug" in statuses else "complete"
        partial_files = sum(item.get("postprocessing", {}).get("partial_annotation_files", 0) for item in annotation_records)
        coverage = "partial" if partial_files or status == "partial_debug" else "complete"
        print("Source restoration complete. Files with unavailable annotation anchors:", partial_files)
        print("Original stories are retained; coverage and unmapped tags are recorded in the annotation manifests.")
        record_stage(RUN_OUTPUT_DIR, "annotation", status, manifests=[str(p) for p in annotation_manifests],
                     annotation_coverage=coverage, partial_annotation_files=partial_files)
    except BaseException:
        record_stage(RUN_OUTPUT_DIR, "annotation", "failed_or_interrupted; review retained outputs")
        raise
    finally:
        persist_run()


In [ ]:
# Final annotated files must pass both preservation QC and Chatter.
if RUN_ANNOTATION:
    from icl_pilot.chat.validation import require_validated_report
    record_stage(RUN_OUTPUT_DIR, "chat_validation_after", "running")
    try:
        final_validation_reports = []
        for path in annotation_manifests:
            annotation_record = json.loads(path.read_text())
            if annotation_record["status"] not in {"annotation_complete", "partial_debug"}:
                raise RuntimeError("Annotation/source-preservation QC has not passed.")
            reference = annotation_record["chatter"]["after"]
            checked = require_validated_report(reference)
            final_validation_reports.append(reference)
            print("Chatter passed after annotation:", reference["manifest"],
                  "| CHAT files:", checked.get("files_checked"))
        record_stage(RUN_OUTPUT_DIR, "chat_validation_after", "complete", reports=final_validation_reports)
    except BaseException:
        record_stage(RUN_OUTPUT_DIR, "chat_validation_after", "failed; final files need review")
        raise
    finally:
        persist_run()


<a id="batchalign-ud"></a>
### 4.1. Batchalign 3 English UD annotation

Run after TalkTag and its final Chatter check. For saved CHAT, select
`MODE="morphosyntax_only"`, set `CHAT_INPUT_DIRS`, run common setup, then
these two cells. Generation, segmentation and TalkTag are skipped in that mode.
For fresh real-plus-synthetic annotation use `from_chat` and shared preparation
so both receive new TalkTag codes before UD. The UD-only route preserves existing
inline codes and skips shared error-code removal; use it only when that is intended.
Use the same pinned settings for real and synthetic narratives.

The isolated CPU runtime uses Batchalign 3 (package 0.10.7), Stanza 1.15.0
and checksum-pinned English models. Only tokenize/MWT/POS/lemma/dependency
processors are loaded. Retokenization and utterance splitting are disabled;
no speech recognition, grammatical correction or traditional CLAN parser runs.
Results caching is bypassed, and each selected file must be accounted for.

Inputs are copied into immutable snapshots. Existing `%mor`, `%gra`, `%umor`
and `%ugra` are removed only from prepared worker copies so annotation is fresh.
Removal counts are recorded. Final copies replace morphology/dependencies for all speakers
with UD `%mor`/`%gra`, retaining exact source words, case, punctuation, TalkTag
codes, headers, boundaries and all non-morphological tiers. CHI coverage is
recorded separately. Previous tiers remain in
the snapshots. Native Batchalign output, including its added provenance comment,
is retained separately; final files retain the original record layout.

Chatter validates prepared inputs and final outputs with full alignment checks.
Missing files/tiers, changed content or failed validation stop processing and
retain diagnostics. Utterances containing only unintelligible/non-speech
material are recorded separately when no morphology is produced. Successful
software checks do not establish linguistic tag accuracy; manual review remains
needed. No IPSyn/DSS score or feature table is calculated in this block.

Final files: `RUN_OUTPUT_DIR/morphosyntax/annotated/`. Runtime dependencies and
model storage are separate from generation/TalkTag; cached models can be reused.
See [UD annotation guide](../docs/guides/batchalign_annotation.md).

The setup cell pins the catalog and every model download to immutable revisions,
retaining their reviewed checksums. The original smoke ZIP is still usable.
After a resource-download failure, rerun this setup cell and the cells below
in the same Colab kernel; existing generations and TalkTag outputs are reused.

Batchalign may replace an earlier tool-version comment from the original corpus.
The merger restores that exact source comment and records the new backend stamp
in its manifest. All words, codes, other headers and boundaries remain checked.


In [ ]:
# Exact reviewed upgrades keep the original smoke ZIP usable in an existing Colab runtime.
if globals().get("RUN_MORPHOSYNTAX", False):
    import hashlib, importlib
    from icl_pilot.generation.recovery import atomic_json
    _ud_updates = {'configs/annotation/batchalign.json': {'before': 'bc3f48cb163d6a684c40ddbda1435c1515bef118eed4bbc10834cf6f0ae9b7ee', 'after': 'eaf10a88d56a144886916b1f41185c32b037c2e2ead5557035b30c5917791e60', 'replacements': [['  ],\n  "resources": {\n    "version": "1.15.0",\n    "url": "https://raw.githubusercontent.com/stanfordnlp/stanza-resources/main/resources_1.15.0.json",\n    "sha256": "b9a397f594aa68b6b09806b0f6f8c95b8f1dd4d1f4459a9ce1553016200d234a"\n  },\n  "models": {\n    "en/tokenize/combined_nocharlm.pt": {\n      "md5": "764639a5f65cc42d84b294566ffa6215",\n      "url": "https://huggingface.co/stanfordnlp/stanza-en/resolve/v1.15.0/models/tokenize/combined_nocharlm.pt",\n      "sha256": "becae178f59cd8adba655a9dc8b3cac68311e1706da0a331706c7804f0d0c42d"\n    },\n    "en/mwt/gum.pt": {\n      "md5": "85857cef3a7031b5d83b3e123ded0f27",\n      "url": "https://huggingface.co/stanfordnlp/stanza-en/resolve/v1.15.0/models/mwt/gum.pt",\n      "sha256": "eac3f5c64015d8e07906a22a456e5835d2da8053e118c4b1deddecfff1be5b1f"\n    },\n    "en/pos/combined_charlm.pt": {\n      "md5": "a31dbf7269152bc85c3c79c5318ae848",\n      "url": "https://huggingface.co/stanfordnlp/stanza-en/resolve/v1.15.0/models/pos/combined_charlm.pt",\n      "sha256": "c3be39f76a12d58b2e8387475db3915a93a4e4a01867d536306fc84e0218b154"\n    },\n    "en/backward_charlm/1billion.pt": {\n      "md5": "1405948b125b4264fd17509d1b6175ca",\n      "url": "https://huggingface.co/stanfordnlp/stanza-en/resolve/v1.15.0/models/backward_charlm/1billion.pt",\n      "sha256": "5d8cb2306650e4fc2b4f7517cc0fb33b38269e05bbf8d05e712e481471052225"\n    },\n    "en/forward_charlm/1billion.pt": {\n      "md5": "468b3377455fa0311565d46865f55afb",\n      "url": "https://huggingface.co/stanfordnlp/stanza-en/resolve/v1.15.0/models/forward_charlm/1billion.pt",\n      "sha256": "8a32ac2226dda8ea5d30f2a15fe3f835639977f299cfda315ce1b8a656596fd2"\n    },\n    "en/pretrain/conll17.pt": {\n      "md5": "c339580492002fb1f759537dfbc57fb2",\n      "url": "https://huggingface.co/stanfordnlp/stanza-en/resolve/v1.15.0/models/pretrain/conll17.pt",\n      "sha256": "edde4c195f91ba45aab3f3a6254202011f284559da8a5ead5e62ce522f819c5e"\n    },\n    "en/lemma/combined_nocharlm.pt": {\n      "md5": "c1919a2fd49702ddb93338c3a06c8b18",\n      "url": "https://huggingface.co/stanfordnlp/stanza-en/resolve/v1.15.0/models/lemma/combined_nocharlm.pt",\n      "sha256": "23aa62d98759d04eac42b2e1262a9e1f09e0ca31e99d0cd9965c36acf271f3a4"\n    },\n    "en/depparse/combined_charlm.pt": {\n      "md5": "993a0eb712cefb881c684df8d9934f15",\n      "url": "https://huggingface.co/stanfordnlp/stanza-en/resolve/v1.15.0/models/depparse/combined_charlm.pt",\n      "sha256": "e2b22ebadcfd35998f64f2beec01faa6a3854d582ba55d60dc412a1db0010752"\n    }\n  },\n', '  ],\n  "resources": {\n    "version": "1.15.0",\n    "revision": "6ddbdc143649e4065dbb769aa661373e8e284a44",\n    "url": "https://raw.githubusercontent.com/stanfordnlp/stanza-resources/6ddbdc143649e4065dbb769aa661373e8e284a44/resources_1.15.0.json",\n    "sha256": "b9a397f594aa68b6b09806b0f6f8c95b8f1dd4d1f4459a9ce1553016200d234a"\n  },\n  "models_revision": "93906d2c3e3641dbebb0e8739cdc870e4031332d",\n  "models": {\n    "en/tokenize/combined_nocharlm.pt": {\n      "md5": "764639a5f65cc42d84b294566ffa6215",\n      "url": "https://huggingface.co/stanfordnlp/stanza-en/resolve/93906d2c3e3641dbebb0e8739cdc870e4031332d/models/tokenize/combined_nocharlm.pt",\n      "sha256": "becae178f59cd8adba655a9dc8b3cac68311e1706da0a331706c7804f0d0c42d"\n    },\n    "en/mwt/gum.pt": {\n      "md5": "85857cef3a7031b5d83b3e123ded0f27",\n      "url": "https://huggingface.co/stanfordnlp/stanza-en/resolve/93906d2c3e3641dbebb0e8739cdc870e4031332d/models/mwt/gum.pt",\n      "sha256": "eac3f5c64015d8e07906a22a456e5835d2da8053e118c4b1deddecfff1be5b1f"\n    },\n    "en/pos/combined_charlm.pt": {\n      "md5": "a31dbf7269152bc85c3c79c5318ae848",\n      "url": "https://huggingface.co/stanfordnlp/stanza-en/resolve/93906d2c3e3641dbebb0e8739cdc870e4031332d/models/pos/combined_charlm.pt",\n      "sha256": "c3be39f76a12d58b2e8387475db3915a93a4e4a01867d536306fc84e0218b154"\n    },\n    "en/backward_charlm/1billion.pt": {\n      "md5": "1405948b125b4264fd17509d1b6175ca",\n      "url": "https://huggingface.co/stanfordnlp/stanza-en/resolve/93906d2c3e3641dbebb0e8739cdc870e4031332d/models/backward_charlm/1billion.pt",\n      "sha256": "5d8cb2306650e4fc2b4f7517cc0fb33b38269e05bbf8d05e712e481471052225"\n    },\n    "en/forward_charlm/1billion.pt": {\n      "md5": "468b3377455fa0311565d46865f55afb",\n      "url": "https://huggingface.co/stanfordnlp/stanza-en/resolve/93906d2c3e3641dbebb0e8739cdc870e4031332d/models/forward_charlm/1billion.pt",\n      "sha256": "8a32ac2226dda8ea5d30f2a15fe3f835639977f299cfda315ce1b8a656596fd2"\n    },\n    "en/pretrain/conll17.pt": {\n      "md5": "c339580492002fb1f759537dfbc57fb2",\n      "url": "https://huggingface.co/stanfordnlp/stanza-en/resolve/93906d2c3e3641dbebb0e8739cdc870e4031332d/models/pretrain/conll17.pt",\n      "sha256": "edde4c195f91ba45aab3f3a6254202011f284559da8a5ead5e62ce522f819c5e"\n    },\n    "en/lemma/combined_nocharlm.pt": {\n      "md5": "c1919a2fd49702ddb93338c3a06c8b18",\n      "url": "https://huggingface.co/stanfordnlp/stanza-en/resolve/93906d2c3e3641dbebb0e8739cdc870e4031332d/models/lemma/combined_nocharlm.pt",\n      "sha256": "23aa62d98759d04eac42b2e1262a9e1f09e0ca31e99d0cd9965c36acf271f3a4"\n    },\n    "en/depparse/combined_charlm.pt": {\n      "md5": "993a0eb712cefb881c684df8d9934f15",\n      "url": "https://huggingface.co/stanfordnlp/stanza-en/resolve/93906d2c3e3641dbebb0e8739cdc870e4031332d/models/depparse/combined_charlm.pt",\n      "sha256": "e2b22ebadcfd35998f64f2beec01faa6a3854d582ba55d60dc412a1db0010752"\n    }\n  },\n']]}, 'src/icl_pilot/chat/morphosyntax.py': {'before': 'cd1d89df42b020e4f602200f00f786035ecf71c6109dfe334bf4796ac487574f', 'after': '347f6c2d1ac3bfed567a93380b44c04bb4559570bc2900953bd2e3588af4a845', 'replacements': [['ROOT = Path(__file__).resolve().parents[3]\nDEFAULT_CONFIG = ROOT / "configs/annotation/batchalign.json"\nMORPH_TIERS = {"%mor", "%gra", "%umor", "%ugra"}\n\n\ndef load_morphosyntax_config(path=DEFAULT_CONFIG):\n', 'ROOT = Path(__file__).resolve().parents[3]\nDEFAULT_CONFIG = ROOT / "configs/annotation/batchalign.json"\nMORPH_TIERS = {"%mor", "%gra", "%umor", "%ugra"}\n_SOURCE_UD_PROVENANCE = re.compile(\n    r"batchalign(?:3)? [a-f0-9]{7,40} \\| morphosyntax: "\n    r"stanza:\\d+\\.\\d+\\.\\d+:noretok:native-mwt\\d+ \\| [0-9T:Z.+-]+")\n\n\ndef load_morphosyntax_config(path=DEFAULT_CONFIG):\n'], ['    if (len(config.get("models", {})) != 8 or not config.get("source_sha256")\n            or not re.fullmatch(r"[a-f0-9]{64}", config.get("resources", {}).get("sha256", ""))):\n        raise ValueError("Annotation requires model/catalog/source provenance")\n    for name, spec in config["models"].items():\n        if (not name.startswith("en/") or ".." in Path(name).parts\n                or not re.fullmatch(r"[a-f0-9]{32}", spec.get("md5", ""))\n                or not re.fullmatch(r"[a-f0-9]{64}", spec.get("sha256", ""))\n                or not spec.get("url", "").startswith("https://huggingface.co/stanfordnlp/stanza-en/resolve/v1.15.0/models/")):\n            raise ValueError("Invalid English Stanza model pin")\n    return config\n\n', '    if (len(config.get("models", {})) != 8 or not config.get("source_sha256")\n            or not re.fullmatch(r"[a-f0-9]{64}", config.get("resources", {}).get("sha256", ""))):\n        raise ValueError("Annotation requires model/catalog/source provenance")\n    resources = config["resources"]\n    if (resources.get("version") != config["stanza_version"]\n            or not re.fullmatch(r"[a-f0-9]{40}", resources.get("revision", ""))\n            or resources.get("url") != "https://raw.githubusercontent.com/stanfordnlp/stanza-resources/"\n                + resources["revision"] + "/resources_" + resources["version"] + ".json"\n            or not re.fullmatch(r"[a-f0-9]{40}", config.get("models_revision", ""))):\n        raise ValueError("Annotation downloads require immutable catalog/model revisions")\n    model_prefix = ("https://huggingface.co/stanfordnlp/stanza-en/resolve/"\n                    + config["models_revision"] + "/models/")\n    for name, spec in config["models"].items():\n        if (not name.startswith("en/") or ".." in Path(name).parts\n                or not re.fullmatch(r"[a-f0-9]{32}", spec.get("md5", ""))\n                or not re.fullmatch(r"[a-f0-9]{64}", spec.get("sha256", ""))\n                or spec.get("url") != model_prefix + name[3:]):\n            raise ValueError("Invalid English Stanza model pin")\n    return config\n\n'], ['        temporary = path.with_suffix(path.suffix + ".downloading")\n        with urlopen(spec["url"], timeout=60) as source, temporary.open("wb") as out:\n            shutil.copyfileobj(source, out)\n        if file_digest(temporary, algorithm) != spec[algorithm]:\n            raise ValueError("Annotation download checksum mismatch: " + str(temporary))\n        temporary.rename(path)\n    return root\n\n', '        temporary = path.with_suffix(path.suffix + ".downloading")\n        with urlopen(spec["url"], timeout=60) as source, temporary.open("wb") as out:\n            shutil.copyfileobj(source, out)\n        actual = file_digest(temporary, algorithm)\n        if actual != spec[algorithm]:\n            raise ValueError("Annotation download checksum mismatch: " + str(temporary)\n                             + "; expected " + spec[algorithm] + "; received " + actual\n                             + "; source " + spec["url"])\n        temporary.rename(path)\n    return root\n\n'], ['\ndef check_morphosyntax_output(source, output):\n    original = _protected(source)\n    projected, added, index = [], [], 0\n    for tier, body in _protected(output):\n        if index < len(original) and (tier, body) == original[index]:\n            projected.append((tier, body))\n            index += 1\n', '\ndef check_morphosyntax_output(source, output):\n    original = _protected(source)\n    projected, added, restored, index = [], [], [], 0\n\n    def restore_source_provenance(next_record=None):\n        nonlocal index\n        # Batchalign replaces its earlier tool-version comment. The final\n        # merger copies the complete original header, so account explicitly\n        # for that metadata-only omission in the raw backend output.\n        while (index < len(original) and original[index] != next_record\n               and original[index][0] == "@Comment"\n               and _SOURCE_UD_PROVENANCE.fullmatch(original[index][1])):\n            projected.append(original[index])\n            restored.append(original[index][1])\n            index += 1\n\n    for tier, body in _protected(output):\n        restore_source_provenance((tier, body))\n        if index < len(original) and (tier, body) == original[index]:\n            projected.append((tier, body))\n            index += 1\n'], ['            added.append(body)\n        else:\n            raise ValueError("Batchalign changed headers, words, codes, punctuation, order or utterance boundaries")\n    if original != projected or len(added) > 1:\n        raise ValueError("Batchalign changed headers, words, codes, punctuation, order or utterance boundaries")\n    blocks = _speaker_blocks(output)\n', '            added.append(body)\n        else:\n            raise ValueError("Batchalign changed headers, words, codes, punctuation, order or utterance boundaries")\n    restore_source_provenance()\n    if original != projected or len(added) > 1:\n        raise ValueError("Batchalign changed headers, words, codes, punctuation, order or utterance boundaries")\n    blocks = _speaker_blocks(output)\n'], ['            excluded.append({"speaker_utterance_index": index, "speaker": block["speaker"]})\n    return {"target_utterances": len(_target_blocks(output)), "tagged_utterances": tagged,\n            "all_speaker_utterances": len(blocks),\n            "nonlexical_utterance_indices": excluded, "native_added_provenance": added}\n\n\ndef merge_morphosyntax_tiers(source, raw):\n', '            excluded.append({"speaker_utterance_index": index, "speaker": block["speaker"]})\n    return {"target_utterances": len(_target_blocks(output)), "tagged_utterances": tagged,\n            "all_speaker_utterances": len(blocks),\n            "nonlexical_utterance_indices": excluded, "native_added_provenance": added,\n            "restored_source_provenance": restored}\n\n\ndef merge_morphosyntax_tiers(source, raw):\n'], ['            manifest["outputs"].append({"relative_path": item["relative_path"], "sha256": sha256_file(target),\n                                        "raw_sha256": sha256_file(raw_path), **check_morphosyntax_output(source, final),\n                                        "native_added_provenance": raw_qc["native_added_provenance"],\n                                        "text_fingerprint": final_fingerprint})\n        report = validation.validate_chat_directories({"annotated": annotated}, output / "validation/after", validator=selected)\n        manifest["chatter"]["after"] = validation.validation_reference(report)\n', '            manifest["outputs"].append({"relative_path": item["relative_path"], "sha256": sha256_file(target),\n                                        "raw_sha256": sha256_file(raw_path), **check_morphosyntax_output(source, final),\n                                        "native_added_provenance": raw_qc["native_added_provenance"],\n                                        "restored_source_provenance": raw_qc["restored_source_provenance"],\n                                        "text_fingerprint": final_fingerprint})\n        report = validation.validate_chat_directories({"annotated": annotated}, output / "validation/after", validator=selected)\n        manifest["chatter"]["after"] = validation.validation_reference(report)\n']], 'previous': {'06cfc3843ed324ebff1821f9176c8dda0c68a4ffda529312c966e998f62e4f70': [['ROOT = Path(__file__).resolve().parents[3]\nDEFAULT_CONFIG = ROOT / "configs/annotation/batchalign.json"\nMORPH_TIERS = {"%mor", "%gra", "%umor", "%ugra"}\n\n\ndef load_morphosyntax_config(path=DEFAULT_CONFIG):\n', 'ROOT = Path(__file__).resolve().parents[3]\nDEFAULT_CONFIG = ROOT / "configs/annotation/batchalign.json"\nMORPH_TIERS = {"%mor", "%gra", "%umor", "%ugra"}\n_SOURCE_UD_PROVENANCE = re.compile(\n    r"batchalign(?:3)? [a-f0-9]{7,40} \\| morphosyntax: "\n    r"stanza:\\d+\\.\\d+\\.\\d+:noretok:native-mwt\\d+ \\| [0-9T:Z.+-]+")\n\n\ndef load_morphosyntax_config(path=DEFAULT_CONFIG):\n'], ['\ndef check_morphosyntax_output(source, output):\n    original = _protected(source)\n    projected, added, index = [], [], 0\n    for tier, body in _protected(output):\n        if index < len(original) and (tier, body) == original[index]:\n            projected.append((tier, body))\n            index += 1\n', '\ndef check_morphosyntax_output(source, output):\n    original = _protected(source)\n    projected, added, restored, index = [], [], [], 0\n\n    def restore_source_provenance(next_record=None):\n        nonlocal index\n        # Batchalign replaces its earlier tool-version comment. The final\n        # merger copies the complete original header, so account explicitly\n        # for that metadata-only omission in the raw backend output.\n        while (index < len(original) and original[index] != next_record\n               and original[index][0] == "@Comment"\n               and _SOURCE_UD_PROVENANCE.fullmatch(original[index][1])):\n            projected.append(original[index])\n            restored.append(original[index][1])\n            index += 1\n\n    for tier, body in _protected(output):\n        restore_source_provenance((tier, body))\n        if index < len(original) and (tier, body) == original[index]:\n            projected.append((tier, body))\n            index += 1\n'], ['            added.append(body)\n        else:\n            raise ValueError("Batchalign changed headers, words, codes, punctuation, order or utterance boundaries")\n    if original != projected or len(added) > 1:\n        raise ValueError("Batchalign changed headers, words, codes, punctuation, order or utterance boundaries")\n    blocks = _speaker_blocks(output)\n', '            added.append(body)\n        else:\n            raise ValueError("Batchalign changed headers, words, codes, punctuation, order or utterance boundaries")\n    restore_source_provenance()\n    if original != projected or len(added) > 1:\n        raise ValueError("Batchalign changed headers, words, codes, punctuation, order or utterance boundaries")\n    blocks = _speaker_blocks(output)\n'], ['            excluded.append({"speaker_utterance_index": index, "speaker": block["speaker"]})\n    return {"target_utterances": len(_target_blocks(output)), "tagged_utterances": tagged,\n            "all_speaker_utterances": len(blocks),\n            "nonlexical_utterance_indices": excluded, "native_added_provenance": added}\n\n\ndef merge_morphosyntax_tiers(source, raw):\n', '            excluded.append({"speaker_utterance_index": index, "speaker": block["speaker"]})\n    return {"target_utterances": len(_target_blocks(output)), "tagged_utterances": tagged,\n            "all_speaker_utterances": len(blocks),\n            "nonlexical_utterance_indices": excluded, "native_added_provenance": added,\n            "restored_source_provenance": restored}\n\n\ndef merge_morphosyntax_tiers(source, raw):\n'], ['            manifest["outputs"].append({"relative_path": item["relative_path"], "sha256": sha256_file(target),\n                                        "raw_sha256": sha256_file(raw_path), **check_morphosyntax_output(source, final),\n                                        "native_added_provenance": raw_qc["native_added_provenance"],\n                                        "text_fingerprint": final_fingerprint})\n        report = validation.validate_chat_directories({"annotated": annotated}, output / "validation/after", validator=selected)\n        manifest["chatter"]["after"] = validation.validation_reference(report)\n', '            manifest["outputs"].append({"relative_path": item["relative_path"], "sha256": sha256_file(target),\n                                        "raw_sha256": sha256_file(raw_path), **check_morphosyntax_output(source, final),\n                                        "native_added_provenance": raw_qc["native_added_provenance"],\n                                        "restored_source_provenance": raw_qc["restored_source_provenance"],\n                                        "text_fingerprint": final_fingerprint})\n        report = validation.validate_chat_directories({"annotated": annotated}, output / "validation/after", validator=selected)\n        manifest["chatter"]["after"] = validation.validation_reference(report)\n']]}}, 'src/icl_pilot/chat/morphosyntax_worker.py': {'before': 'fe5e5b0fe8713a78332bc460a30aa89e231d42d127828b7e0b2fcd286538a5e9', 'after': 'cf07e931d1fd461e17146b60544e3f8c23228f41db5dba5a785de253f110d7fa', 'replacements': [['        for block in iter(lambda: handle.read(1024 * 1024), b""):\n            digest.update(block)\n    return digest.hexdigest()\n\n\ndef run_worker(request_path, response_path):\n', '        for block in iter(lambda: handle.read(1024 * 1024), b""):\n            digest.update(block)\n    return digest.hexdigest()\n\n\ndef pin_catalog_refresh(stanza, config):\n    """Keep Batchalign\'s startup refresh on the reviewed catalog revision."""\n    common = stanza.resources.common\n    resources = config["resources"]\n    if common.DEFAULT_RESOURCES_VERSION != resources["version"]:\n        raise ValueError("Stanza catalog refresh version differs from the pinned configuration")\n    # The hash-checked Batchalign backend reads this constant at startup.\n    # Its normal refresh may run, but it can only retrieve the same catalog.\n    common.DEFAULT_RESOURCES_URL = resources["url"].rsplit("/", 1)[0]\n\n\ndef run_worker(request_path, response_path):\n'], ['                                        "depparse": "combined_charlm"})\n            return settings\n\n    backend = EnglishUDBackend(lang="en", retokenize=False, device=config["device"])\n    if model_inventory() != models:\n        raise ValueError("Stanza resources changed while initializing annotation")\n', '                                        "depparse": "combined_charlm"})\n            return settings\n\n    import stanza\n    pin_catalog_refresh(stanza, config)\n    backend = EnglishUDBackend(lang="en", retokenize=False, device=config["device"])\n    if model_inventory() != models:\n        raise ValueError("Stanza resources changed while initializing annotation")\n'], ['               "model_files": models, "resources_sha256": config["resources"]["sha256"],\n               "retokenize": False, "resegment": False, "device": config["device"],\n               "workers": config["workers"], "cache_policy": "Bypass", "torch_threads": torch.get_num_threads(),\n               "adapter": "EnglishUDBackend._lang_config: explicit UD-only processors; downloads disabled"}\n    response_path.write_text(json.dumps({"schema_version": 1, "completed_files": completed, "runtime": runtime,\n                                         "request_sha256": file_digest(request_path)}, indent=2) + "\\n")\n\n', '               "model_files": models, "resources_sha256": config["resources"]["sha256"],\n               "retokenize": False, "resegment": False, "device": config["device"],\n               "workers": config["workers"], "cache_policy": "Bypass", "torch_threads": torch.get_num_threads(),\n               "catalog_refresh_url": config["resources"]["url"],\n               "adapter": "EnglishUDBackend._lang_config: explicit UD-only processors; downloads disabled; startup catalog refresh pinned"}\n    response_path.write_text(json.dumps({"schema_version": 1, "completed_files": completed, "runtime": runtime,\n                                         "request_sha256": file_digest(request_path)}, indent=2) + "\\n")\n\n']]}}
    _ud_pending = []
    _ud_states = {}
    for _relative, _spec in _ud_updates.items():
        _path = REPO_DIR / _relative
        _data = _path.read_text()
        _digest = hashlib.sha256(_data.encode()).hexdigest()
        if _digest == _spec["after"]:
            _ud_states[_relative] = "already_current"
            continue
        if _digest != _spec["before"] and _digest not in _spec.get("previous", {}):
            raise RuntimeError("UD resource helper differs from both reviewed versions: " + _relative)
        for _before, _after in _spec.get("previous", {}).get(_digest, _spec["replacements"]):
            if _data.count(_before) != 1:
                raise RuntimeError("UD upgrade context changed: " + _relative)
            _data = _data.replace(_before, _after, 1)
        if hashlib.sha256(_data.encode()).hexdigest() != _spec["after"]:
            raise RuntimeError("UD upgrade checksum failed: " + _relative)
        _ud_pending.append((_path, _data))
        _ud_states[_relative] = "upgraded_from_reviewed_bundle"
    # Validate all files before changing any helper/configuration.
    for _path, _data in _ud_pending:
        _temporary = _path.with_name(_path.name + ".reviewed-upgrade")
        _temporary.write_text(_data)
        _temporary.replace(_path)
    importlib.invalidate_caches()
    from icl_pilot.chat import morphosyntax_worker as _ud_worker, morphosyntax as _ud_module
    importlib.reload(_ud_worker)
    importlib.reload(_ud_module)
    _ud_config = _ud_module.load_morphosyntax_config(MORPHOSYNTAX_CONFIG_PATH)
    _ud_policy = dict(schema_version=1, policy="immutable-stanza-resources-v1",
        resources_revision=_ud_config["resources"]["revision"],
        resources_sha256=_ud_config["resources"]["sha256"],
        models_revision=_ud_config["models_revision"],
        source_sha256={name: spec["after"] for name, spec in _ud_updates.items()},
        upgrades=_ud_states)
    _ud_policy_path = RUN_OUTPUT_DIR / "morphosyntax_resource_policy.json"
    atomic_json(_ud_policy_path, _ud_policy)
    record_stage(RUN_OUTPUT_DIR, "morphosyntax_resource_policy", "complete", **_ud_policy)
    print("UD resource policy: immutable catalog/model revisions; original checksums retained.")

# END REVIEWED UD RESOURCE UPGRADE
if globals().get("RUN_MORPHOSYNTAX", False):
    from icl_pilot.chat.morphosyntax import (load_morphosyntax_config,
        install_morphosyntax_runtime, prepare_morphosyntax_models, annotate_morphosyntax)
    from icl_pilot.chat.validation import require_validated_report
    if RUN_ANNOTATION:
        from icl_pilot.chat.annotation_preparation import preparation_reference, checked_talktag_collections
        if len(annotation_manifests) != 1:
            raise RuntimeError("Use the single combined TalkTag manifest for this run.")
        morphosyntax_upstream = preparation_reference(annotation_manifests[0])
        morphosyntax_inputs, _, _ = checked_talktag_collections(morphosyntax_upstream)
    else:
        if MODE != "morphosyntax_only":
            raise RuntimeError("Use morphosyntax_only with saved CHAT_INPUT_DIRS.")
        morphosyntax_inputs = annotation_inputs(chat_roots, ANNOTATION_STAGES)
        morphosyntax_upstream = None  # Existing fresh TalkTag codes must remain intact on this UD-only route.
    morphosyntax_settings = load_morphosyntax_config(MORPHOSYNTAX_CONFIG_PATH)
    if INSTALL_DEPENDENCIES:
        morphosyntax_python = install_morphosyntax_runtime(MORPHOSYNTAX_ENV, MORPHOSYNTAX_CONFIG_PATH)
    else:
        morphosyntax_python = MORPHOSYNTAX_ENV / ("Scripts/python.exe" if sys.platform == "win32" else "bin/python")
    morphosyntax_model_dir = prepare_morphosyntax_models(MORPHOSYNTAX_MODELS_DIR,
        MORPHOSYNTAX_CONFIG_PATH, download=INSTALL_DEPENDENCIES)
    print("UD inputs:", {key: str(path) for key, path in morphosyntax_inputs.items()})
    print("UD versions:", {key: morphosyntax_settings[key] for key in ("version", "stanza_version", "torch_version")})
else:
    print("Batchalign UD annotation skipped.")


In [ ]:
if globals().get("RUN_MORPHOSYNTAX", False):
    from icl_pilot.chat import validation as ud_validation
    record_stage(RUN_OUTPUT_DIR, "morphosyntax", "running")
    try:
        ud_validator = ud_validation.ensure_chatter(executable=globals().get("CHATTER_EXECUTABLE"),
            config_path=CHATTER_CONFIG_PATH, install=INSTALL_DEPENDENCIES)
        morphosyntax_manifest = annotate_morphosyntax(morphosyntax_inputs, RUN_OUTPUT_DIR / "morphosyntax",
            python=morphosyntax_python, model_dir=morphosyntax_model_dir,
            config_path=MORPHOSYNTAX_CONFIG_PATH, validator=ud_validator,
            test_only=json.loads((RUN_OUTPUT_DIR / "pipeline_manifest.json").read_text())["test_only"],
            upstream_annotation=morphosyntax_upstream)
        morphosyntax_record = json.loads(morphosyntax_manifest.read_text())
        morphosyntax_output_dir = morphosyntax_manifest.parent / morphosyntax_record["annotated_output_directory"]
        record_stage(RUN_OUTPUT_DIR, "morphosyntax", "complete", manifest=str(morphosyntax_manifest),
            output=str(morphosyntax_output_dir))
        print("Validated UD-annotated CHAT:", morphosyntax_output_dir)
    except BaseException:
        record_stage(RUN_OUTPUT_DIR, "morphosyntax", "failed_or_interrupted; review retained outputs")
        raise
    finally:
        persist_run()


## 5. QC summary and exports

The run folder includes generation CSVs/journal/design when generated here,
review-only HTML/CSV/JSON reports when requested,
Batchalign boundary/runtime records, converted CHAT manifests, shared preparation
retained synthetic-original snapshots, real-source identities/hashes/PIDs,
deletion audits/collection provenance, derived annotation input snapshots, raw/postprocessed CHAT,
TalkTag logs/reports, Batchalign UD source/prepared snapshots, raw/final CHAT,
model/runtime hashes and validation/QC manifests. All travel together in the export.
Saved external input paths/hashes remain recorded; retain the original generation
run's provenance when starting from its CSV or CHAT files.

This is a processing record, not an evaluation result. Continue evaluation in
the separate Python notebook or R Markdown after annotation review and the
appropriate measurement/feature-extraction steps. Annotation ends with the
validated Batchalign UD output; traditional MOR/MEGRASP and IPSyn are not run. The bounded Qwen generation-to-export Colab smoke run passed; other models,
full-corpus execution and linguistic annotation accuracy still require evaluation.

<a id="annotation-retention"></a>
### 3.6. Verify final text and retain one annotated copy

Run after TalkTag and, when enabled, UD. The export block also invokes this check.
Words, speaker order, main-tier boundaries and headers are checked against the
preparation baseline; UD additionally preserves all existing inline annotations.
Intended annotation removals/additions and new morphology are allowed. Every final
file is also recorded with its complete SHA-256 and per-utterance fingerprints.

Only a complete checked chain can be compacted. The block removes run-owned
source snapshots, prepared copies, raw model CHAT and (after UD) intermediate
TalkTag CHAT. It retains final CHAT, manifests, runtime records, deletion audits,
validation reports and diagnostics. External inputs, original corpus ZIPs and
generation CSVs/journals remain available. Drive synchronization removes only the
same checksum-matching intermediate files. Failed or incomplete runs are preserved.

After cleanup, use `verify_retained_annotations(reference, run_root=...)` with the
reference in `pipeline_manifest.json`. Earlier stage reports remain historical;
their deleted inputs are not needed by this final check. To rerun annotation,
start a fresh processing run from the saved input folders. To recover a particular
real original, use the ZIP/member reference with `restore_original_member`; the
documented ENNI sex-header corrections belong to the study input view.


In [ ]:
def finalize_annotation_retention():
    if not globals().get("LEAN_ANNOTATION", False) or not RUN_ANNOTATION:
        return
    from icl_pilot.chat.retention import finalize_annotation_run
    from icl_pilot.chat.corpus_archives import DEFAULT_CONFIG as DEFAULT_RETENTION_CONFIG
    stages = json.loads((RUN_OUTPUT_DIR / "pipeline_manifest.json").read_text())["stages"]
    required = ["annotation_preparation", "annotation", "chat_validation_after"]
    if globals().get("RUN_MORPHOSYNTAX", False):
        required.append("morphosyntax")
    if any(stages.get(name, {}).get("status") != "complete" for name in required):
        print("Annotation incomplete: working copies and diagnostics retained.")
        return
    if len(stages["annotation"]["manifests"]) != 1:
        raise ValueError("Retention requires the one shared TalkTag invocation.")
    reference = finalize_annotation_run(RUN_OUTPUT_DIR, annotation_manifest=stages["annotation"]["manifests"][0],
        morphosyntax_manifest=stages.get("morphosyntax", {}).get("manifest")
            if globals().get("RUN_MORPHOSYNTAX", False) else None,
        archive_paths=globals().get("ORIGINAL_CORPUS_ARCHIVES", {}), project_root=REPO_DIR,
        config_path=globals().get("RETENTION_CONFIG_PATH", DEFAULT_RETENTION_CONFIG))
    record_stage(RUN_OUTPUT_DIR, "annotation_retention", "complete", **reference)
    persist_run()
    print("Final annotations and preservation receipt:", reference["manifest"])

finalize_annotation_retention()


In [ ]:
if REAL_DATA_PILOT:
    from icl_pilot.chat.retention import verify_retained_annotations
    from icl_pilot.generation.recovery import atomic_json
    _summary = json.loads((RUN_OUTPUT_DIR / "pipeline_manifest.json").read_text())
    _required = ["real_annotation_pilot", "annotation_preparation", "annotation_preservation_policy",
        "chat_validation_before", "annotation", "chat_validation_after", "morphosyntax_resource_policy",
        "morphosyntax", "annotation_retention"]
    assert _summary["test_only"] is True
    assert all(_summary["stages"].get(name, {}).get("status") == "complete" for name in _required)
    assert _summary["stages"]["generation"]["status"] == "skipped"
    assert _summary["stages"]["segmentation"]["status"] == "skipped"
    _receipt = verify_retained_annotations(_summary["stages"]["annotation_retention"], run_root=RUN_OUTPUT_DIR)
    _expected = {"real_enni/" + child["path"][len("chat/"): ] for child in real_pilot_manifest["children"]}
    _actual = {item["path"].split("/annotated/", 1)[1] for item in _receipt["outputs"]}
    assert _actual == _expected and len(_receipt["outputs"]) == 6
    _ud = json.loads(Path(_summary["stages"]["morphosyntax"]["manifest"]).read_text())
    report = dict(test_only=True, profile="real_enni_pilot", input_children=6,
        clinical_counts=real_pilot_manifest["clinical_counts"], reference_narratives=36,
        final_annotated_files=6, chi_utterances=sum(item["target_utterances"] for item in _ud["outputs"]),
        tagged_chi_utterances=sum(item["tagged_utterances"] for item in _ud["outputs"]),
        original_boundaries_retained=True,
        annotation_coverage=_summary["stages"]["annotation"].get("annotation_coverage", "unknown"),
        partial_annotation_files=_summary["stages"]["annotation"].get("partial_annotation_files", 0),
        checks_passed=_required, linguistic_accuracy_review="pending")
    atomic_json(RUN_OUTPUT_DIR / "smoke_verification.json", report)
    record_stage(RUN_OUTPUT_DIR, "smoke_verification", "complete", report=str(RUN_OUTPUT_DIR / "smoke_verification.json"))
    print("Real ENNI annotation-pilot checks passed; exporting next.")
    print(json.dumps(report, indent=2))


<a id="domain-features"></a>
## Domain-specific features — KIDEVAL English UD

Run the **repository** cell, then start here to use saved final annotations.
After a new annotation run, leave `FEATURE_INPUTS=[]` to select its completed
UD manifest automatically. To combine real and synthetic data, list their final
UD manifests or processing ZIPs together. Gillam is an external classifier test
set and is not an input to this ENNI domain-analysis block.

[Settings](#feature-settings) | [Extraction](#feature-extraction) | [Export](#feature-export)

This section assembles six-story units in A1–A3, B1–B3 order: one per real child
or synthetic model/run/pair/replicate/stage. It preserves words, boundaries and
annotations. Only audio links and timing bullets are removed from the derived
analysis bundles. Final annotated source files stay unchanged and hash-verified.

CLAN runs with the supported English UD script (`+lengu`), CHI-only scoring,
raw morphological counts and the default DSS sample requirement. IPSyn and
normative-database comparisons are disabled. Incomplete bundles remain flagged
rows with missing features; no generation QC flags remove stories. Missing
TalkTag coverage remains explicit, and DSS sentence scores are unavailable for
partially assessed units. Category totals retain their native meaning.

Outputs include combined and separate real/synthetic feature CSVs, the full raw
CLAN report, DSS category tables, an inventory and a checksum manifest. Native
word/utterance error columns stay in the raw report only; custom error measures
excluding preposition errors and the separate R statistics are not run here.
Pilot rows carry `test_only` and must not enter study results.

`RUN_DOMAIN_FEATURES=False` leaves this section inactive during Run All.

<a id="feature-settings"></a>
### Feature inputs and settings

Set the run switch, select completed final-UD manifests or processing ZIPs, and
run support, runtime and extraction below. Saved-input execution needs no models,
generation, TalkTag or Batchalign inference. CPU is sufficient. The native CLAN
tools are built once from the checksum-pinned official Unix source archive.
Colab defaults to Drive for independent results; an active processing run keeps
features under its own output directory so the normal processing ZIP includes them.

For the original smoke upload, upload
`ICL-PILOT_domain_features_support_20261102.zip` alongside its original bundle.
The existing bootstrap is needed only to unpack the original project in a fresh
smoke runtime. Do not run the generation settings just to extract saved features.

In [ ]:
RUN_DOMAIN_FEATURES = False
FEATURE_INPUTS = []  # e.g. [Path("/content/real_processing.zip"), Path("/content/synthetic_processing.zip")]
FEATURE_OUTPUT_DIR = None  # Optional fresh directory; existing results are never overwritten
FEATURE_OUTPUT_ROOT = Path("/content/drive/MyDrive/ICL-PILOT/domain_features") if IS_COLAB else REPO_DIR / "data/generated/domain_features"
CLAN_ENV = Path("/content/icl_pilot_clan") if IS_COLAB else REPO_DIR / ".cache/clan"
FEATURE_SUPPORT_ZIP = None  # Updated checkout already includes the helpers
MAKE_FEATURE_ARCHIVE = True
DOWNLOAD_FEATURE_ARCHIVE = False  # VS Code can download through Colab Contents
feature_manifest = None
if not RUN_DOMAIN_FEATURES:
    print("Domain feature extraction disabled.")

In [ ]:
if RUN_DOMAIN_FEATURES and FEATURE_SUPPORT_ZIP is not None:
    import hashlib, importlib, json, zipfile
    _feature_zip = Path(FEATURE_SUPPORT_ZIP)
    if not _feature_zip.is_file():
        raise FileNotFoundError("Upload ICL-PILOT_domain_features_support_20261102.zip first.")
    if hashlib.sha256(_feature_zip.read_bytes()).hexdigest() != "e3348fbd99b5289b4366642b96f2197803c7063ec83654a5eed71421b4555414":
        raise ValueError("Domain feature support ZIP differs from the reviewed package.")
    with zipfile.ZipFile(_feature_zip) as _z:
        _spec = json.loads(_z.read("domain_features_support_manifest.json"))
        if len(_z.namelist()) != len(_spec) + 1 or set(_z.namelist()) != set(_spec) | {"domain_features_support_manifest.json"}:
            raise ValueError("Unexpected or duplicate feature support members.")
        _pending = []
        for _name, _item in _spec.items():
            _relative = Path(_name)
            if _relative.is_absolute() or ".." in _relative.parts or "\\" in _name:
                raise ValueError("Unsafe feature support member.")
            _target = REPO_DIR / _relative
            if _target.is_symlink() or any(p.is_symlink() for p in _target.parents):
                raise ValueError("Feature support destination contains a symlink.")
            _data = _z.read(_name)
            if hashlib.sha256(_data).hexdigest() != _item["sha256"]:
                raise ValueError("Feature support member checksum mismatch: " + _name)
            if _target.exists():
                _actual = hashlib.sha256(_target.read_bytes()).hexdigest()
                if _actual not in {_item["sha256"], _item["before"]}:
                    raise ValueError("Existing unreviewed file preserved: " + str(_target))
                if _actual == _item["sha256"]:
                    continue
            _pending.append((_target,_data))
        for _target,_data in _pending:
            _target.parent.mkdir(parents=True,exist_ok=True)
            _target.write_bytes(_data)
    importlib.invalidate_caches()
    import sys
    for _module in ("icl_pilot.features.kideval_runtime", "icl_pilot.features.kideval"):
        if _module in sys.modules:
            importlib.reload(sys.modules[_module])
    print("Verified domain feature support files:",len(_spec))

In [ ]:
if RUN_DOMAIN_FEATURES:
    import json
    import pandas as pd
    if IS_COLAB and (str(FEATURE_OUTPUT_ROOT).startswith("/content/drive/") or
                     any(str(p).startswith("/content/drive/") for p in FEATURE_INPUTS)):
        from google.colab import drive
        if not Path("/content/drive/MyDrive").is_dir():
            drive.mount("/content/drive")
    from icl_pilot.features.kideval_runtime import install_clan
    from icl_pilot.features.kideval import extract_kideval
    clan_runtime_manifest = install_clan(REPO_DIR,CLAN_ENV)
    print("Verified native CLAN runtime:",clan_runtime_manifest)

<a id="feature-extraction"></a>
### Extract features from final annotations

Each execution creates a fresh output directory. Inputs must have completed UD
manifests and matching final-file checksums. An incomplete bundle is kept in the
inventory with missing measurements. Duplicate child or bundle/story inputs are
rejected, including selecting a final manifest and its ZIP together.

In [ ]:
if RUN_DOMAIN_FEATURES:
    from datetime import datetime, timezone
    _feature_inputs = [Path(p).expanduser() for p in FEATURE_INPUTS]
    if not _feature_inputs:
        _current_ud = globals().get("morphosyntax_manifest")
        if _current_ud is None or not Path(_current_ud).is_file():
            raise ValueError("Set FEATURE_INPUTS to completed final-UD manifests or processing ZIPs.")
        _feature_inputs = [Path(_current_ud)]
    _stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
    _active_run = Path(RUN_OUTPUT_DIR) if "RUN_OUTPUT_DIR" in globals() else None
    _in_processing_run = (_active_run is not None and (_active_run / "pipeline_manifest.json").is_file()
                          and not FEATURE_INPUTS)
    _feature_root = _active_run / "domain_features" if _in_processing_run else Path(FEATURE_OUTPUT_ROOT)
    feature_output_dir = Path(FEATURE_OUTPUT_DIR) if FEATURE_OUTPUT_DIR is not None else _feature_root / _stamp
    feature_manifest = extract_kideval(_feature_inputs,feature_output_dir,
        project_root=REPO_DIR,runtime=clan_runtime_manifest)
    if _in_processing_run:
        from icl_pilot.pipeline import record_stage
        record_stage(_active_run,"domain_features","complete",manifest=str(feature_manifest),output=str(feature_output_dir))
        if "persist_run" in globals():
            persist_run()
    print("Feature outputs:",feature_output_dir)
    print("Feature manifest:",feature_manifest)

In [ ]:
if RUN_DOMAIN_FEATURES and feature_manifest is not None:
    _features = pd.read_csv(feature_manifest.parent / "kideval_features.csv")
    display(_features[["source","Group","model_key","stage","n_stories","feature_status",
                       "TalkTag_Coverage","test_only"]].fillna(""))
    print("Rows:",len(_features),"| Complete bundles:",int(_features.complete_six_stories.sum()))
    print("Combined CSV:",feature_manifest.parent / "kideval_features.csv")
    print("Separate CSVs: real_kideval_features.csv, synthetic_kideval_features.csv")

<a id="feature-export"></a>
### Export domain features

The feature archive contains derived bundles, native reports, CSVs, validation
and provenance. The original real corpus ZIPs are referenced and are not copied
into this archive. If this was part of the current annotation run, the regular
processing export below also includes its feature directory.

In [ ]:
if RUN_DOMAIN_FEATURES and feature_manifest is not None and MAKE_FEATURE_ARCHIVE:
    from icl_pilot.pipeline import package_run
    feature_archive = package_run(feature_manifest.parent)
    print("Domain feature archive:",feature_archive)
    if IS_COLAB:
        import shutil
        _download = Path("/content") / feature_archive.name
        shutil.copy2(feature_archive,_download)
        print("Colab Contents download:",_download.name)
        if DOWNLOAD_FEATURE_ARCHIVE:
            from google.colab import files
            files.download(str(_download))

In [ ]:
if "finalize_annotation_retention" in globals():
    finalize_annotation_retention()
elif globals().get("LEAN_ANNOTATION", False) and RUN_ANNOTATION:
    raise RuntimeError("Run the final text-verification/retention block before exporting annotations.")
summary = json.loads((RUN_OUTPUT_DIR / "pipeline_manifest.json").read_text())
print(json.dumps(summary["stages"], indent=2))
persist_run()
if MAKE_ARCHIVE:
    archive_path = package_run(RUN_OUTPUT_DIR)
    print("Complete processing archive:", archive_path)
    if IS_COLAB:
        import shutil
        download_archive_path = Path("/content") / archive_path.name
        shutil.copy2(archive_path, download_archive_path)
        print("Colab Contents download:", download_archive_path.name)
    persist_run()
    if DOWNLOAD_ARCHIVE:
        from google.colab import files
        files.download(str(archive_path))

if RELEASE_COLAB_RUNTIME and IS_COLAB:
    from google.colab import runtime
    runtime.unassign()

<a id="framework-evaluation"></a>
## 6. Framework-based evaluation — independent saved-text route

Run the **repository** cell near the top, then start here. Generation settings,
model loading, CHAT conversion and annotation cells are not required. This
section reads saved narrative text; R analyses stay in their separate notebook.
After new generation, select the saved CSVs from **all three generators** here.
Keep generators separate and use the common real ENNI reference and real splits.

[Settings and inputs](#framework-settings) | [Corpus metrics](#framework-corpus) |
[Text quality](#framework-quality) | [Classification](#framework-classification) |
[Ratio experiment](#framework-ratios) | [Export](#framework-export)

`RUN_FRAMEWORK_EVALUATION=False` makes the whole section inactive during Run All.
When enabled, input checks and a training plan run first; BERT fitting and
GPT-2/MiniLM inference each require their own switch. Smoke-test data cannot enter
scientific evaluation. QC flags remain review-only and do not remove stories.

For later sessions, set `EVAL_RESUME_DIR` to the saved evaluation directory and
run repository, settings, support, session, runtime and input-check cells before
jumping to a selected block. Inputs and source code must still match the saved
checksums. Colab writes results incrementally to `EVAL_OUTPUT_ROOT` on Drive;
local runs default to `results/framework_based_evaluation/` in the checkout.
Completed classifier fits are verified and reused; interrupted partial
fits remain preserved and require the existing recovery workflow or a fresh run.

<a id="framework-settings"></a>
### 6.1. Evaluation settings and saved inputs

Choose either `EVAL_GENERATION_CSVS` (recommended) or a prebuilt
`EVAL_TEXT_RECORDS_CSV`. For each generator select its aggregate CSV **or** its
four cohort CSVs. Selecting both produces duplicate identities and is rejected.
Gillam remains a separate external test input. All paths refer to this kernel's
filesystem; mount Drive first when selecting files there.

The isolated evaluation environment preserves the generation and annotation
dependencies. Basic corpus metrics need only the base runtime. To run BERT or
GPT-2/MiniLM, enable `INSTALL_FRAMEWORK_TOOLKIT` in addition to the relevant run
switch. Setup reuses the environment on later visits; no kernel restart is needed.

In [ ]:
# Independent settings: these do not alter the generation/annotation settings.
RUN_FRAMEWORK_EVALUATION = False
EVAL_GENERATION_CSVS = {}  # One labelled selection per generator, e.g.:
# {
#   "llama3_1_8b": [Path("/content/drive/MyDrive/llama_run/all_cohorts_rotating_story_generations.csv")],
#   "qwen3_8b": [Path("/content/drive/MyDrive/qwen_run/all_cohorts_rotating_story_generations.csv")],
#   "gemma2_9b": [Path("/content/drive/MyDrive/gemma_run/all_cohorts_rotating_story_generations.csv")],
# }
EVAL_TEXT_RECORDS_CSV = None  # Alternative: an existing common-policy long-form export
EVAL_GILLAM_CSV = REPO_DIR / "data/inputs/gillam/external_story_records.csv"
EVAL_PROTOCOL_PATH = REPO_DIR / "configs/evaluation/protocol.json"
EVAL_RESUME_DIR = None  # Existing evaluation directory; reuses its saved inputs/ratios
EVAL_OUTPUT_ROOT = Path("/content/drive/MyDrive/ICL-PILOT/framework_based_evaluation") if IS_COLAB else None
EVAL_SYNTHETIC_REAL_RATIOS = (1, 10)
EVAL_ENV = Path("/content/icl_pilot_framework_env") if IS_COLAB else REPO_DIR / ".venv-framework"
INSTALL_FRAMEWORK_RUNTIME = True
INSTALL_FRAMEWORK_TOOLKIT = False  # Required for the model inference blocks
RUN_EVAL_DESCRIPTIVE = True
RUN_EVAL_TEXT_QUALITY = False
RUN_EVAL_CLASSIFICATION = False
RUN_EVAL_SUMMARIES = False  # True to summarise previously completed fits without fitting
PREPARE_EVAL_RATIO_PLAN = False
RUN_EVAL_RATIOS = False  # Separate from RUN_EVAL_CLASSIFICATION
RUN_EVAL_RATIO_SUMMARIES = False
EVAL_SAVE_TO_DRIVE = IS_COLAB
EVAL_DRIVE_ROOT = Path("/content/drive/MyDrive/ICL-PILOT/framework_based_evaluation")
EVAL_DOWNLOAD_ARCHIVE = False  # VS Code users can download via Colab Contents instead
FRAMEWORK_SUPPORT_ZIP = None  # Current checkout already includes the helpers/inputs
framework_session = None
framework_output = None
if not RUN_FRAMEWORK_EVALUATION:
    print("Framework evaluation disabled; no evaluation inputs, models or fits will be loaded.")

In [ ]:
# The original smoke upload can receive this exact reviewed evaluation package.
if RUN_FRAMEWORK_EVALUATION and FRAMEWORK_SUPPORT_ZIP is not None:
    import hashlib, importlib, json, zipfile
    _support_zip = Path(FRAMEWORK_SUPPORT_ZIP)
    if not _support_zip.is_file():
        raise FileNotFoundError("Upload ICL-PILOT_framework_support_20261102.zip to this runtime first.")
    if hashlib.sha256(_support_zip.read_bytes()).hexdigest() != "8ebf595b931c5252d5e069466ab27604506aff1d0097ebf736dcf0e1263a63b1":
        raise ValueError("Framework support ZIP differs from the reviewed package.")
    with zipfile.ZipFile(_support_zip) as _z:
        _spec = json.loads(_z.read("framework_support_manifest.json"))
        if len(_z.namelist()) != len(_spec) + 1 or set(_z.namelist()) != set(_spec) | {"framework_support_manifest.json"}:
            raise ValueError("Unexpected or duplicate framework support members.")
        _pending = []
        for _name, _item in _spec.items():
            _relative = Path(_name)
            if _relative.is_absolute() or ".." in _relative.parts or "\\" in _name:
                raise ValueError("Unsafe framework support member.")
            _target = REPO_DIR / _relative
            if _target.is_symlink() or any(p.is_symlink() for p in _target.parents):
                raise ValueError("Framework support destination contains a symlink.")
            _data = _z.read(_name)
            if hashlib.sha256(_data).hexdigest() != _item["sha256"]:
                raise ValueError("Framework support member checksum mismatch: " + _name)
            if _target.exists():
                _actual = hashlib.sha256(_target.read_bytes()).hexdigest()
                if _actual not in {_item["sha256"], _item["before"], *_item.get("accepted_previous", [])}:
                    raise ValueError("Existing unreviewed file preserved: " + str(_target))
                if _actual == _item["sha256"]:
                    continue
            _pending.append((_target, _data))
        for _target, _data in _pending:
            _target.parent.mkdir(parents=True, exist_ok=True)
            _target.write_bytes(_data)
    importlib.invalidate_caches()
    import sys
    for _module in ("icl_pilot.evaluation.runs", "icl_pilot.evaluation.notebook", "icl_pilot.evaluation.notebook_runtime"):
        if _module in sys.modules:
            importlib.reload(sys.modules[_module])
    print("Verified framework support files:", len(_spec))

In [ ]:
if RUN_FRAMEWORK_EVALUATION:
    if not (REPO_DIR / "src/icl_pilot/evaluation/notebook.py").is_file():
        raise RuntimeError("Use the updated checkout or select FRAMEWORK_SUPPORT_ZIP before framework setup.")
    from icl_pilot.evaluation.notebook import create_session, load_session, execute_block, export_session, sha256
    _eval_paths = [*([str(v) for values in EVAL_GENERATION_CSVS.values() for v in
                     (values if isinstance(values, (list, tuple)) else [values])]),
                   str(EVAL_TEXT_RECORDS_CSV or ""), str(EVAL_GILLAM_CSV), str(EVAL_RESUME_DIR or ""), str(EVAL_OUTPUT_ROOT or "")]
    if IS_COLAB and (EVAL_SAVE_TO_DRIVE or any(p.startswith("/content/drive/") for p in _eval_paths)):
        from google.colab import drive
        if not Path("/content/drive/MyDrive").is_dir():
            drive.mount("/content/drive")
    framework_session = create_session(REPO_DIR, generation_csvs=EVAL_GENERATION_CSVS,
        text_records_csv=EVAL_TEXT_RECORDS_CSV, gillam_csv=EVAL_GILLAM_CSV,
        protocol_path=EVAL_PROTOCOL_PATH, resume_dir=EVAL_RESUME_DIR, ratios=EVAL_SYNTHETIC_REAL_RATIOS,
        output_root=EVAL_OUTPUT_ROOT)
    framework_output = framework_session.parent
    print("Evaluation directory:", framework_output)
    print("To return later, set EVAL_RESUME_DIR =", repr(str(framework_output)))

    def show_framework_table(relative, limit=24):
        import csv, html
        from itertools import islice
        from IPython.display import display, HTML
        path = framework_output / relative
        if path.is_file():
            with path.open(newline="", encoding="utf-8") as handle:
                rows = list(islice(csv.reader(handle), limit + 1))
            display(HTML("<p>" + html.escape(relative) + "</p><table>" +
                "".join("<tr>" + "".join("<td>" + html.escape(v) + "</td>" for v in row) + "</tr>" for row in rows) + "</table>"))

    def release_framework_gpu_memory():
        import gc
        for name in ("generation_runtime", "model", "tokenizer"):
            globals().pop(name, None)
        gc.collect()
        if "torch" in globals() and torch.cuda.is_available():
            torch.cuda.empty_cache()

In [ ]:
if RUN_FRAMEWORK_EVALUATION:
    from icl_pilot.evaluation.notebook_runtime import install_runtime, verify_runtime
    if INSTALL_FRAMEWORK_RUNTIME:
        framework_python = install_runtime(EVAL_ENV, toolkit=INSTALL_FRAMEWORK_TOOLKIT)
    else:
        framework_python = EVAL_ENV / ("Scripts/python.exe" if os.name == "nt" else "bin/python")
    verify_runtime(framework_python, toolkit=RUN_EVAL_TEXT_QUALITY or RUN_EVAL_CLASSIFICATION or RUN_EVAL_RATIOS)
    print("Independent evaluation Python:", framework_python)

In [ ]:
if RUN_FRAMEWORK_EVALUATION:
    execute_block(framework_session, "inputs", framework_python)
    show_framework_table("split_audit.csv")
    show_framework_table("classification/training_plan.csv")

<a id="framework-corpus"></a>
### 6.2. Descriptive corpus comparisons

Length, lexical diversity, Jaccard and shared-real TF–IDF cosine are reported
separately per generator. The real comparison rows are shared across generators;
these descriptive comparisons do not establish clinical fidelity. This block
loads no neural model. Metric definitions and OOV diagnostics are saved.

In [ ]:
if RUN_FRAMEWORK_EVALUATION and RUN_EVAL_DESCRIPTIVE:
    execute_block(framework_session, "descriptive", framework_python)
    show_framework_table("descriptive_by_generator_age_group.csv")
    show_framework_table("lexical_overlap_by_generator.csv")

<a id="framework-quality"></a>
### 6.3. Automated text-quality diagnostics — opt in

Set `RUN_EVAL_TEXT_QUALITY=True` and install the toolkit runtime. This downloads
the pinned GPT-2 and MiniLM checkpoints, computes perplexity and Gaussian
mean/covariance Fréchet distance separately per generator, and saves embeddings
and truncation audits. These are corpus diagnostics, not clinical outcomes.

In [ ]:
if RUN_FRAMEWORK_EVALUATION and RUN_EVAL_TEXT_QUALITY:
    release_framework_gpu_memory()
    execute_block(framework_session, "quality", framework_python)
    show_framework_table("text_quality_by_generator.csv")

<a id="framework-classification"></a>
### 6.4. BERT classification and saved-fit summaries — opt in

Five age-stratified grouped folds include all 238 real ENNI children, including
unmatched TD children. Prompt-pair members stay together; only training-ancestor
synthetic bundles enter each fit. Synthetic draws match the real training
age/group/story distribution with replacement. Loss remains unweighted.

The main plan compares real-only, synthetic-only and augmented training, with
shared real-only baseline/update-matched controls and three classifier seeds.
Gillam is external test-only. BERT uses at most 512 tokens including specials;
each fit writes its token/truncation audit. Completed fits are verified before
reuse. ENNI/Gillam predictions are saved at story level and aggregated by child;
cluster-bootstrap intervals remain conditional on the recorded fits and splits.

Set `RUN_EVAL_CLASSIFICATION=True` for GPU fitting. To summarise saved completed
fits only, resume their directory and set `RUN_EVAL_SUMMARIES=True`.

In [ ]:
if RUN_FRAMEWORK_EVALUATION and RUN_EVAL_CLASSIFICATION:
    release_framework_gpu_memory()
    execute_block(framework_session, "classifiers", framework_python)

In [ ]:
if RUN_FRAMEWORK_EVALUATION and (RUN_EVAL_CLASSIFICATION or RUN_EVAL_SUMMARIES):
    execute_block(framework_session, "summary", framework_python)
    show_framework_table("classification/summary/summary_across_rounds_and_seeds.csv")
    show_framework_table("classification/summary/paired_differences_cluster_bootstrap.csv")
    show_framework_table("classification/summary/age_class_recall_summary.csv")

<a id="framework-ratios"></a>
### 6.5. Separate synthetic-to-real ratio experiment — opt in

The default optional levels are 1:1 and 10:1 sampled synthetic:real training
stories. Real folds, classifier seeds and the real-only baseline are shared;
synthetic draw prefixes are nested across levels and common across generators.
Each ratio has its own update-matched real-only control. Verified identical main
fits can be reused. Ratios are prespecified, not selected using ENNI test/Gillam.

`PREPARE_EVAL_RATIO_PLAN=True` writes the plan without fitting.
`RUN_EVAL_RATIOS=True` separately enables ratio fitting. For summaries of saved
ratio fits only, use `RUN_EVAL_RATIO_SUMMARIES=True`. Tables and a child-level
balanced-accuracy curve are exported; the curve itself is descriptive.

In [ ]:
if RUN_FRAMEWORK_EVALUATION and (PREPARE_EVAL_RATIO_PLAN or RUN_EVAL_RATIOS):
    execute_block(framework_session, "ratio-plan", framework_python)
    show_framework_table("ratio_experiment/ratio_training_plan.csv")

In [ ]:
if RUN_FRAMEWORK_EVALUATION and RUN_EVAL_RATIOS:
    release_framework_gpu_memory()
    execute_block(framework_session, "ratios", framework_python)

In [ ]:
if RUN_FRAMEWORK_EVALUATION and (RUN_EVAL_RATIOS or RUN_EVAL_RATIO_SUMMARIES):
    execute_block(framework_session, "ratio-summary", framework_python)
    show_framework_table("ratio_experiment/summary/summary.csv")
    show_framework_table("ratio_experiment/summary/ratio_differences.csv")
    from IPython.display import display, Image
    display(Image(filename=str(framework_output / "ratio_experiment/child_balanced_accuracy_by_ratio.png")))

<a id="framework-export"></a>
### 6.6. Compact verified evaluation export

This export is separate from generation/annotation exports. It includes prepared
text, splits, bootstrap selections, plans, runtime/source/input provenance and all
completed predictions, metrics and figures. Model/trainer directories remain on
disk and are omitted from the portable ZIP. Every ZIP member is checksum-checked.
Copy the ZIP to Drive or download it from the displayed path in Colab Contents.
The runtime remains connected until you verify the download on your computer.

In [ ]:
if RUN_FRAMEWORK_EVALUATION:
    import shutil
    framework_archive = export_session(framework_session)
    print("Verified evaluation ZIP:", framework_archive)
    print("SHA256:", sha256(framework_archive))
    if EVAL_SAVE_TO_DRIVE:
        if not IS_COLAB:
            raise RuntimeError("Select a local export path or use Drive in Colab.")
        EVAL_DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
        _destination = EVAL_DRIVE_ROOT / framework_archive.name
        if _destination.exists() and sha256(_destination) != sha256(framework_archive):
            raise FileExistsError("Existing different Drive ZIP preserved: " + str(_destination))
        shutil.copy2(framework_archive, _destination)
        if sha256(_destination) != sha256(framework_archive):
            raise OSError("Drive evaluation ZIP checksum mismatch.")
        print("Verified Drive ZIP:", _destination)
    if IS_COLAB:
        _contents_copy = Path("/content") / framework_archive.name
        if _contents_copy.exists() and sha256(_contents_copy) != sha256(framework_archive):
            raise FileExistsError("Existing different Contents ZIP preserved.")
        shutil.copy2(framework_archive, _contents_copy)
        if sha256(_contents_copy) != sha256(framework_archive):
            raise OSError("Contents ZIP checksum mismatch.")
        print("Colab Contents download path:", _contents_copy)
        if EVAL_DOWNLOAD_ARCHIVE:
            from google.colab import files
            files.download(str(_contents_copy))